# Moteur de recommandation

> **Description** : ce notebook construit, étape par étape, le moteur de recommandation de l'application. À partir de la sélection de films de l'équipe, complétée par les données TMDB (mots-clés, titre original), chaque film est transformé en une ligne de nombres (genres, année, durée, note, popularité, mots-clés, langue). Un algorithme des plus proches voisins (KNN) trouve ensuite les films qui ressemblent le plus à un film donné.
>
> Chaque étape est expliquée (**Objectif** avant le code, **Constat** après) et les choix sont justifiés. Le code final du moteur sera ensuite repris dans `reco.py`, utilisé par l'application Streamlit.

## 1. Les données

> **Objectif** : charger la sélection de films de l'équipe (`films_selection.csv`), point de départ du catalogue.

In [1]:
import pandas as pd

In [2]:
films = pd.read_csv("../data/films_selection.csv")
print(films.shape)
films.columns

(6092, 13)


Index(['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear',
       'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'title',
       'original_language', 'poster_path', 'sans_titre'],
      dtype='str')

> **Objectif** : vérifier que chaque film n'apparaît qu'une fois (identifiant `tconst`). `duplicated()` compte les doublons ; avec `keep=False`, on affiche toutes les versions des films concernés pour les comparer.

In [3]:
doublons = films.duplicated("tconst")
print(doublons.sum())
films[doublons]

3


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre
4193,tt1756855,movie,Coyote vs. Acme,0,2026.0,103.0,"Adventure,Animation,Comedy",7.3,51477,Coyote vs. Acme,en,NaN,False
4604,tt23468450,movie,Longlegs,0,2024.0,101.0,"Crime,Horror,Mystery",6.5,237152,Longlegs,en,/3wVnnLzX5eObFWzrsy5vJGpMCCI.jpg,False
4807,tt27441481,movie,Coolie,0,2025.0,168.0,"Action,Crime,Drama",6.0,25450,Coolie,ta,/hDntDhxW0uKsCQRaUfcwCyYUXKY.jpg,False


In [4]:
doublons_keep = films.duplicated("tconst", keep=False)
print(doublons_keep.sum())
films[doublons_keep]

6


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre
4192,tt1756855,movie,Coyote vs. Acme,0,2026.0,103.0,"Adventure,Animation,Comedy",7.3,51477,Coyote vs. Acme,en,NaN,False
4193,tt1756855,movie,Coyote vs. Acme,0,2026.0,103.0,"Adventure,Animation,Comedy",7.3,51477,Coyote vs. Acme,en,NaN,False
4603,tt23468450,movie,Longlegs,0,2024.0,101.0,"Crime,Horror,Mystery",6.5,237152,Longlegs,en,NaN,False
4604,tt23468450,movie,Longlegs,0,2024.0,101.0,"Crime,Horror,Mystery",6.5,237152,Longlegs,en,/3wVnnLzX5eObFWzrsy5vJGpMCCI.jpg,False
4806,tt27441481,movie,Coolie,0,2025.0,168.0,"Action,Crime,Drama",6.0,25450,Coolie,ta,/3GUVDQ0arZcZzxRAVfyAxijTKEe.jpg,False
4807,tt27441481,movie,Coolie,0,2025.0,168.0,"Action,Crime,Drama",6.0,25450,Coolie,ta,/hDntDhxW0uKsCQRaUfcwCyYUXKY.jpg,False


In [5]:
films[doublons_keep][["tconst", "title", "poster_path"]]

,tconst,title,poster_path
4192,tt1756855,Coyote vs. Acme,NaN
4193,tt1756855,Coyote vs. Acme,NaN
4603,tt23468450,Longlegs,NaN
4604,tt23468450,Longlegs,/3wVnnLzX5eObFWzrsy5vJGpMCCI.jpg
4806,tt27441481,Coolie,/3GUVDQ0arZcZzxRAVfyAxijTKEe.jpg
4807,tt27441481,Coolie,/hDntDhxW0uKsCQRaUfcwCyYUXKY.jpg


> **Constat** : 3 films apparaissent deux fois ; leurs deux versions ne diffèrent que par l'affiche (`poster_path`) : aucune affiche, une seule, ou deux affiches différentes selon le film.

> **Objectif** : supprimer les doublons avec une règle générique, sans numéro de ligne écrit en dur : si un film apparaît plusieurs fois, on garde une seule version, de préférence une qui a une affiche. Méthode : marquer les versions avec affiche, trier pour les faire passer en premier, puis garder la première ligne de chaque film (`drop_duplicates`). S'il n'y a aucun doublon, ce code ne change rien.

In [6]:
films["a_affiche"] = films["poster_path"].notna()
films["a_affiche"].value_counts()

a_affiche
True     5796
False     296
Name: count, dtype: int64

In [7]:
films = films.sort_values("a_affiche", ascending=False)
films

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre,a_affiche
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold,en,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg,False,True
3893,tt1499658,movie,Horrible Bosses,0,2011.0,98.0,"Comedy,Crime",6.9,493543,Comment tuer son boss ?,en,/uQkUwgyFHAm0jGQERPG6Z9o9Zbj.jpg,False,True
3891,tt14966898,movie,Late Night with the Devil,0,2023.0,93.0,Horror,7.0,156018,Late Night with the Devil,en,/aiAovxn125YXUm1TLqnMgD3znRa.jpg,False,True
3890,tt1496025,movie,Underworld: Awakening,0,2012.0,88.0,"Action,Fantasy,Thriller",6.3,169242,Underworld : Nouvelle Ère,en,/jN0uuc8U6M3sTg9zEaliJV60Stf.jpg,False,True
3889,tt14948432,movie,Red One,0,2024.0,123.0,"Action,Adventure,Comedy",6.2,162948,Red One,en,/ca7pxBRGeVFATZsgAH4Ks06cOs5.jpg,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5047,tt32141377,movie,28 Years Later: The Bone Temple,0,2026.0,109.0,"Horror,Sci-Fi,Thriller",7.2,135844,28 ans plus tard: Le temple des morts,NaN,NaN,False,False
5046,tt32104007,movie,Young Washington,0,2026.0,121.0,"Action,Adventure,Biography",6.8,15195,Young Washington,en,NaN,False,False
5045,tt32080876,movie,Dragon,0,2025.0,157.0,"Comedy,Drama,Romance",7.8,19718,Dragon,en,NaN,False,False
5043,tt32063050,movie,Bridget Jones: Mad About the Boy,0,2025.0,124.0,"Comedy,Drama,Romance",6.5,36180,Bridget Jones : Folle de lui,NaN,NaN,False,False


In [8]:
films = films.drop_duplicates(subset="tconst")

In [9]:
films.shape

(6089, 14)

In [10]:
print(films["tconst"].duplicated().sum())
print(films.index.max())

0
6091


> **Constat** : il reste 6 089 lignes, mais la plus grande étiquette de l'index vaut 6091 : les lignes supprimées ont laissé des « trous » dans l'index. De plus, le tri a changé l'ordre des lignes (les films sans affiche sont à la fin).

> **Objectif** : comprendre, sur un mini-exemple de 3 films, pourquoi ces « trous » sont dangereux. pandas utilise deux numérotations : les *étiquettes* de l'index (utilisées par `loc`, `index`, `idxmax`) et les *positions* 0, 1, 2… (utilisées par `iloc`, les tableaux numpy et les résultats d'un modèle). Ici, `mini` a les étiquettes 0, 1, 3 (comme un tableau dont on aurait supprimé une ligne), et `autre` les étiquettes 0, 1, 2.

In [11]:
mini = pd.DataFrame({"film": ["Alien", "Toy Story", "Matrix"]}, index=[0, 1, 3])
autre = pd.DataFrame({"note": [8.4, 8.3, 8.7]})   # index automatique 0, 1, 2
print(mini)
print(autre)

        film
0      Alien
1  Toy Story
3     Matrix
   note
0   8.4
1   8.3
2   8.7


In [12]:
pd.concat([mini, autre], axis=1)

,film,note
0,Alien,8.4
1,Toy Story,8.3
3,Matrix,NaN
2,NaN,8.7


> **Constat** : `pd.concat(axis=1)` associe les lignes par étiquette, pas par position. Avec des étiquettes différentes (0, 1, 3 d'un côté ; 0, 1, 2 de l'autre), on obtient 4 lignes au lieu de 3 et des `NaN` : Matrix perd sa note, qui part sur une ligne fantôme.

In [13]:
i = mini.index[mini["film"] == "Matrix"][0]
print(i)    # va donner 3
print(autre)
# autre.iloc[[i]]   # provoque volontairement une IndexError (étiquette prise pour une position)

3
   note
0   8.4
1   8.3
2   8.7


> **Constat** : `mini.index[…]` renvoie une *étiquette* (3), alors que `iloc` attend une *position* (0, 1 ou 2) : erreur `IndexError`. Si la position avait existé, on aurait obtenu un autre film, sans message d'erreur.
>
> **Conclusion** : avec des « trous » dans l'index, deux bugs menacent la suite : un `pd.concat` décalerait les lignes et créerait des `NaN` ; un film repéré par son étiquette puis cherché par sa position renverrait le mauvais film. Il faut donc que étiquette = position : on renumérote de 0 à n − 1.

> **Objectif** : réparer l'index. On retire la colonne de travail `a_affiche`, on remet les films dans leur ordre d'origine (`sort_index`), puis on renumérote de 0 à n − 1 (`reset_index(drop=True)`) pour que étiquette = position. La jointure plus loin renumérote aussi les lignes, mais on garde cette étape par sécurité : le code reste juste même si l'on change l'ordre des opérations.

In [14]:
films = films.drop(columns="a_affiche")

In [15]:
films = films.sort_index()

In [16]:
films = films.reset_index(drop=True)  # drop=True jette l’ancien index au lieu d’en faire une nouvelle colonne

In [17]:
print(films.shape)
print(films.index.min(), films.index.max())

(6089, 13)
0 6088


> **Constat** : 6 089 films, avec un index de 0 à 6 088 : étiquette = position.

In [18]:
liste_des_colonnes = list(films.columns)
print(liste_des_colonnes)
print(len(liste_des_colonnes))

['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'title', 'original_language', 'poster_path', 'sans_titre']
13


> **Constat** : classement des 13 colonnes de la sélection.
> - *Décrivent le film* (utiles pour mesurer une ressemblance) : `genres`, `startYear`, `runtimeMinutes`, `averageRating`, `numVotes`, `original_language`.
> - *Affichage* : `title` (titre FR), `primaryTitle` (titre international), `poster_path` (affiche) ; `tconst` est l'identifiant (clé de jointure).
> - *Autres* : `titleType` (toujours « movie ») et `isAdult` (toujours 0) sont constantes après filtrage ; `sans_titre` indique les films sans titre FR.
>
> Il manque une description du *contenu* du film (de quoi il parle) : ce sera le rôle des mots-clés de TMDB.

### Le fichier TMDB

In [19]:
apercu = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", nrows=5)
print(apercu.shape)
apercu.columns

(5, 24)


Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords'],
      dtype='str')

> **Objectif** : ne charger que les colonnes utiles de TMDB (5 sur 24) :
> - `keywords` : les mots-clés du film ;
> - `original_title` : le titre en version originale ;
> - `imdb_id` : l'identifiant IMDb (`tt…`), le même que `tconst` dans notre sélection ; c'est la clé de jointure ;
> - `poster_path` et `popularity` : serviront à départager les doublons de TMDB (affiche d'abord, popularité ensuite).
>
> On ne charge que ces 5 colonnes : moins de mémoire et lecture plus rapide.

In [20]:
colonnes_tmdb = ["imdb_id", "keywords", "original_title", "poster_path", "popularity"]

In [21]:
tmdb = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", usecols=colonnes_tmdb)
tmdb.shape

(1529949, 5)

> **Objectif** : retirer les films TMDB sans identifiant IMDb (`imdb_id` vide). Sans identifiant, ils ne pourraient jamais être joints à notre sélection, dont tous les films viennent d'IMDb : on ne perd donc rien.

In [22]:
tmdb["imdb_id"].isna().sum()

np.int64(836561)

In [23]:
tmdb = tmdb.dropna(subset=["imdb_id"])
tmdb.shape

(693388, 5)

> **Constat** : 836 561 films sur 1 529 949 n'ont pas d'identifiant IMDb (plus de la moitié, surtout des films très confidentiels, absents d'IMDb). Il reste 693 388 films.

> **Objectif** : repérer les films qui apparaissent plusieurs fois dans TMDB (même `imdb_id`), puis regarder de près ceux qui concernent nos films.

In [24]:
# doublons dans tmdb ?
tmdb["imdb_id"].duplicated().sum()

np.int64(5633)

In [25]:
doublons_tmdb = tmdb[tmdb["imdb_id"].duplicated(keep=False)]

In [26]:
doublons_nos_films = doublons_tmdb[doublons_tmdb["imdb_id"].isin(films["tconst"])]
doublons_nos_films["imdb_id"].nunique()

3

In [27]:
doublons_nos_films.sort_values("imdb_id")

,imdb_id,original_title,popularity,poster_path,keywords
508610,tt1756855,Coyote vs. Acme,1.400,NaN,"lawsuit, live action and animation, based on m..."
1429111,tt1756855,Coyote v. Acme,2.069,NaN,NaN
529903,tt23468450,Longlegs,0.000,NaN,"fbi, serial killer, female protagonist, occult"
1295936,tt23468450,Longlegs,1.400,/3wVnnLzX5eObFWzrsy5vJGpMCCI.jpg,NaN
612321,tt27441481,கூலி,0.000,/3GUVDQ0arZcZzxRAVfyAxijTKEe.jpg,NaN
1325326,tt27441481,Thalaivar 171,0.618,/hDntDhxW0uKsCQRaUfcwCyYUXKY.jpg,NaN


> **Constat** : 5 633 identifiants `imdb_id` apparaissent plusieurs fois, dont 3 concernent nos films : ce sont exactement les 3 films en double de la sélection. La sélection a donc hérité des doublons de TMDB lors de sa propre jointure. Souvent, une version est une fiche complète et l'autre une fiche vide ou provisoire (popularité proche de 0, titre de travail comme « Thalaivar 171 »).

> **Objectif** : garder une seule version par `imdb_id` : d'abord celle qui a une affiche, puis, à égalité, la plus populaire (tri sur deux colonnes, puis `drop_duplicates`). Si aucune version n'a d'affiche, toutes valent `False` pour le premier critère : c'est alors la popularité qui décide.

In [28]:
tmdb["a_affiche"] = tmdb["poster_path"].notna()

In [29]:
tmdb = tmdb.sort_values(["a_affiche", "popularity"], ascending=False)

In [30]:
tmdb = tmdb.drop_duplicates(subset="imdb_id")

In [31]:
tmdb = tmdb.drop(columns="a_affiche")
print(tmdb.shape, tmdb["imdb_id"].duplicated().sum())

(687755, 5) 0


> **Constat** : il reste 687 755 films, sans doublon. *Coyote vs. Acme*, qui n'a d'affiche dans aucune version, reste sans affiche.

### La jointure

> **Objectif** : ajouter à chacun de nos films ses mots-clés (`keywords`) et son titre VO (`original_title`), en associant `tconst` (sélection) et `imdb_id` (TMDB). On n'emporte pas `poster_path`, déjà présent dans la sélection (sinon pandas créerait `poster_path_x` et `poster_path_y`). Jointure à gauche : on garde tous nos films, même ceux absents de TMDB.

In [32]:
tmdb_a_joindre = tmdb[["imdb_id", "keywords", "original_title"]]

In [33]:
films = films.merge(tmdb_a_joindre, left_on="tconst", right_on="imdb_id", how="left")
films.shape

(6089, 16)

In [34]:
# Combien de films sont absents de TMDB ? Après un left join, un imdb_id vide veut dire « pas trouvé ».
films["imdb_id"].isna().sum()

np.int64(180)

In [35]:
films = films.drop(columns="imdb_id")
films.shape

(6089, 15)

> **Constat** : toujours 6 089 lignes (aucun film perdu ni dupliqué, grâce au dédoublonnage de TMDB). 180 films sont absents de TMDB : presque tous sortis en 2024-2026, après l'extraction du fichier TMDB (instantané figé) ; ils n'ont ni mots-clés, ni titre VO, ni affiche.

> **Objectif** : compter les films sans mots-clés et voir de quelles années ils datent.

In [36]:
# Films sans mots-clés
films["keywords"].isna().sum()

np.int64(489)

In [37]:
filtre_sans_mots_cles = films["keywords"].isna()
films[filtre_sans_mots_cles].head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre,keywords,original_title
72,tt0068519,movie,Gentlemen of Fortune,0,1971.0,88.0,"Comedy,Crime,Drama",8.3,14681,Les Gentilshommes de la chance,ru,/owZ2KQ2x99XqMprSvFNeTdmCBk8.jpg,False,NaN,Джентльмены удачи
484,tt0085470,movie,Easy Money,0,1983.0,95.0,Comedy,6.2,11530,Hold-up en jupons,en,/nREZhCfkUgl6XqMZAvd6DCbVj2c.jpg,False,NaN,Easy Money
782,tt0094357,movie,A Better Tomorrow II,0,1987.0,105.0,"Action,Crime,Drama",7.1,13154,Le Syndicat du crime 2,zh,/e8352Dw1SeM4rNAGYDZVkQ0aenz.jpg,False,NaN,英雄本色2
791,tt0094716,movie,The Beast of War,0,1988.0,111.0,"Adventure,Drama,War",7.3,11849,La Bête de guerre,en,/ag0oez7oBuk8xCQ5pvb4pSj544Z.jpg,False,NaN,The Beast of War
878,tt0097235,movie,The Dream Team,0,1989.0,113.0,"Comedy,Crime,Thriller",6.5,16360,Une journée de fous,en,/lBVjTP8Cb66GBabUBQaBUtV6OqD.jpg,False,NaN,The Dream Team


In [38]:
films[filtre_sans_mots_cles]["startYear"].value_counts().head(10)

startYear
2025.0    126
2026.0     88
2024.0     77
2023.0     35
2022.0     26
2019.0     13
2020.0     10
2012.0      9
2021.0      9
2014.0      9
Name: count, dtype: int64

> **Constat** : 489 films, surtout récents (2025 : 126 ; 2026 : 88 ; 2024 : 77 ; 2023 : 35 ; 2022 : 26). Ce sont les 180 films absents de TMDB, plus des films présents dans TMDB mais que la communauté n'a pas encore décrits par des mots-clés.

> **Objectif** : donner aux films sans mots-clés leurs genres comme mots-clés, en minuscules et au même format que TMDB (« action, crime »).
>
> Pourquoi ne pas les laisser vides ? Intuition : un film sans mots-clés ne pourrait être comparé aux autres que par ses genres et ses chiffres ; il risque de fausser les ressemblances (à vérifier lors du calcul des distances).

In [39]:
genres_texte = films["genres"].str.lower().str.replace(",", ", ")

In [40]:
print(genres_texte.shape)
genres_texte.head()

(6089,)


0     comedy, fantasy, romance
1     drama, mystery, thriller
2             sci-fi, thriller
3    adventure, drama, romance
4       drama, horror, mystery
Name: genres, dtype: str

In [41]:
films["keywords"] = films["keywords"].fillna(genres_texte)

In [42]:
films["keywords"].isna().sum()

np.int64(0)

> **Constat** : il ne reste aucun film sans mots-clés.

> **Objectif** : enregistrer le résultat dans `data/films_prepares.csv`. La préparation (lire les 674 Mo de TMDB, nettoyer, joindre) n'est ainsi faite qu'une fois : l'appli lira directement ce petit fichier, au lieu de tout refaire à chaque démarrage.

In [43]:
films.to_csv("../data/films_prepares.csv", index=False)

In [44]:
# Vérification
pd.read_csv("../data/films_prepares.csv").shape

(6089, 15)

> **Constat** : le fichier se relit correctement : 6 089 films, 15 colonnes.

## 2. Un film en nombres

> **Objectif** : un calcul de distance ne fonctionne qu'avec des nombres. On transforme donc chaque film en une ligne de nombres : d'abord ses genres (en colonnes 0/1), puis ses caractéristiques numériques (année, durée, note, nombre de votes).

> **Remarque** : on s'attend à trouver plus de 7 genres. Le filtre garde un film dès qu'il possède *au moins un* des 7 genres visés (Action, Adventure, Comedy, Thriller, Horror, Sci-Fi, War), mais ses autres genres (Drama, Crime, Romance…) restent dans la colonne `genres`.

> **Objectif** : lister tous les genres présents, en découpant chaque texte `genres` sur les virgules.

In [45]:
liste_genres = []
for texte in films["genres"]:
    for genre in texte.split(","):
        if genre not in liste_genres:
            liste_genres.append(genre)

In [46]:
print("Nb de genres :", len(liste_genres))

Nb de genres : 21


In [47]:
sorted(liste_genres)

['Action',
 'Adventure',
 'Animation',
 'Biography',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'History',
 'Horror',
 'Music',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Sport',
 'Thriller',
 'War',
 'Western']

> **Objectif** : transformer la colonne texte `genres` en une colonne par genre, avec 1 si le film a ce genre et 0 sinon. `str.get_dummies(sep=",")` fait ce travail en une ligne : il découpe chaque texte sur les virgules (comme notre boucle) et crée automatiquement une colonne par genre rencontré. Cela va donc créer 21 colonnes (avec des 0 et des 1).

In [48]:
genres = films["genres"].str.get_dummies(sep=",")
genres.shape

(6089, 21)

In [49]:
genres.head()

,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western
0,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
3,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,...,1,0,0,1,0,0,0,0,0,0


> **Constat** : 6 089 lignes (une par film) et 21 colonnes (une par genre), comme le nombre trouvé avec la boucle. Chaque ligne contient des 1 pour les genres du film et des 0 partout ailleurs.

In [50]:
nb_genres = genres.sum(axis=1)
nb_genres.value_counts()

3    4534
2    1313
1     242
Name: count, dtype: int64

> **Constat** : la grande majorité des films ont 3 genres (4 534), 1 313 en ont 2 et seulement 242 n'en ont qu'un. Deux films se ressembleront donc par plusieurs genres à la fois, pas seulement par un genre principal.

> **Objectif** : comparer avec `pd.get_dummies`, qui crée aussi des colonnes 0/1, mais sans découper le texte sur les virgules.

In [51]:
pd.get_dummies(films["genres"]).shape

(6089, 321)

> **Constat** : 321 colonnes au lieu de 21. `pd.get_dummies` prend chaque texte entier comme une seule catégorie : « Action,Crime,Drama » et « Action,Drama » deviennent deux colonnes distinctes. Deux films partageant Action et Drama n'auraient alors rien en commun. `str.get_dummies(sep=",")` découpe d'abord le texte, et chaque genre devient une colonne : c'est ce qu'il nous faut, puisqu'un film peut avoir plusieurs genres.

> **Remarque (vocabulaire)** :
> - *one-hot encoding* : chaque film appartient à **une seule** catégorie, donc il y a exactement un 1 par ligne (c'est ce que fait `pd.get_dummies`, par exemple pour la langue originale) ;
> - *multi-hot encoding* : un film peut appartenir à **plusieurs** catégories à la fois, donc une ligne peut contenir plusieurs 1 (c'est ce que fait `str.get_dummies(sep=",")` pour les genres : jusqu'à trois 1 par film).

> **Objectif** : regrouper les 4 caractéristiques chiffrées des films : année, durée, note et nombre de votes.

In [52]:
numeriques = films[["startYear", "runtimeMinutes", "averageRating", "numVotes"]]
numeriques.info()

<class 'pandas.DataFrame'>
RangeIndex: 6089 entries, 0 to 6088
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   startYear       6089 non-null   float64
 1   runtimeMinutes  6089 non-null   float64
 2   averageRating   6089 non-null   float64
 3   numVotes        6089 non-null   int64  
dtypes: float64(3), int64(1)
memory usage: 190.4 KB


> **Objectif** : afficher l'année et la durée en nombres entiers (1971 au lieu de 1971.0). On vérifie d'abord qu'il n'y a aucune valeur manquante, car un entier ne peut pas valoir NaN.

In [53]:
films[["startYear", "runtimeMinutes"]].isna().sum()

startYear         0
runtimeMinutes    0
dtype: int64

> **Constat** : aucune valeur manquante, la conversion en entiers est possible. Cela ne change rien aux calculs : c'est seulement plus lisible.

In [54]:
films["startYear"] = films["startYear"].astype(int)
films["runtimeMinutes"] = films["runtimeMinutes"].astype(int)

In [55]:
numeriques = films[["startYear", "runtimeMinutes", "averageRating", "numVotes"]]
numeriques.info()

<class 'pandas.DataFrame'>
RangeIndex: 6089 entries, 0 to 6088
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   startYear       6089 non-null   int64  
 1   runtimeMinutes  6089 non-null   int64  
 2   averageRating   6089 non-null   float64
 3   numVotes        6089 non-null   int64  
dtypes: float64(1), int64(3)
memory usage: 190.4 KB


> **Objectif** : regarder les ordres de grandeur de ces 4 colonnes (minimum, maximum, moyenne…) pour repérer celles qui poseront problème dans un calcul de distance.

In [56]:
numeriques.describe()

,startYear,runtimeMinutes,averageRating,numVotes
count,6089.000000,6089.000000,6089.000000,6.089000e+03
mean,2005.981113,110.473148,6.840631,1.282149e+05
std,13.980926,21.137044,0.586811,2.185311e+05
min,1970.000000,63.000000,6.000000,1.000400e+04
25%,1997.000000,97.000000,6.400000,2.034800e+04
50%,2009.000000,107.000000,6.800000,4.833900e+04
75%,2017.000000,120.000000,7.200000,1.372810e+05
max,2026.000000,566.000000,9.100000,3.250069e+06


> **Constat** : les 4 colonnes ont des échelles très différentes :
> - année : de 1970 à 2026 (écart de 56) ;
> - durée : de 63 à 566 minutes ; la moitié des films durent entre 97 et 120 minutes, le maximum (566) est donc très isolé ;
> - note : de 6 à 9,1 seulement (le filtre a gardé les films notés au moins 6) ;
> - votes : de 10 004 à 3 250 069.
>
> Le nombre de votes est le plus problématique. Ses valeurs sont énormes, et très déséquilibrées : la moyenne (128 215) est presque trois fois plus grande que la médiane (48 339), signe que quelques films à plusieurs millions de votes tirent la moyenne vers le haut.

> **Objectif** : remplacer le nombre de votes par son logarithme en base 10 (10 000 → 4, 100 000 → 5, 1 000 000 → 6). L'écart entre les films très populaires et les autres devient beaucoup plus raisonnable.

In [57]:
import numpy as np

numeriques["log_votes"] = np.log10(numeriques["numVotes"])

> **Remarque** : ajouter une colonne à `numeriques` ne modifie pas `films`. Depuis pandas 3, une table extraite d'une autre (`films[[...]]`) se comporte toujours comme une copie indépendante. Avec pandas 2, il fallait souvent ajouter `.copy()` pour éviter un avertissement.

In [58]:
print(films.columns)
print(numeriques.columns)

Index(['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear',
       'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'title',
       'original_language', 'poster_path', 'sans_titre', 'keywords',
       'original_title'],
      dtype='str')
Index(['startYear', 'runtimeMinutes', 'averageRating', 'numVotes',
       'log_votes'],
      dtype='str')


> **Constat** : `log_votes` a bien été ajoutée à `numeriques`, et `films` est resté inchangé.

> **Objectif** : vérifier l'effet du logarithme sur l'échelle des votes : le minimum et le maximum doivent être proches de 4 et de 6,5.

In [59]:
numeriques[["log_votes", "numVotes"]].describe()

,log_votes,numVotes
count,6089.000000,6.089000e+03
mean,4.758121,1.282149e+05
std,0.525505,2.185311e+05
min,4.000174,1.000400e+04
25%,4.308522,2.034800e+04
50%,4.684298,4.833900e+04
75%,5.137610,1.372810e+05
max,6.511893,3.250069e+06


> **Constat** : avec le logarithme, les votes vont de 4,0 (10 004 votes) à 6,51 (3 250 069 votes). La moyenne (4,76) est maintenant proche de la médiane (4,68) : les quelques films à plusieurs millions de votes n'écrasent plus les autres.

> **Objectif** (curiosité, non indispensable) : trier les films par nombre de votes, pour voir d'un coup les moins votés (début du tableau) et les plus votés (fin du tableau).

In [60]:
films.sort_values("numVotes")[["title", "startYear", "averageRating", "numVotes"]]

,title,startYear,averageRating,numVotes
5116,Jeux d'enfants,2014,6.3,10004
1843,Train de vie,1998,7.6,10009
3549,Black Is King,2020,6.0,10010
81,Baby Cart: le sabre de la vengeance,1972,7.6,10018
2745,Poultrygeist: Night of the Chicken Dead,2006,6.0,10018
...,...,...,...,...
1728,Matrix,1999,8.7,2293711
2943,Interstellar,2014,8.7,2633501
1745,Fight Club,1999,8.8,2683106
3702,Inception,2010,8.8,2889022


> **Constat** : les films les moins votés ont tous à peine plus de 10 000 votes : c'est le seuil minimum fixé par le filtre de l'équipe. Les plus votés sont des films très connus (Matrix, Interstellar, Fight Club, Inception, The Dark Knight) et aussi très bien notés (de 8,7 à 9,1). The Dark Knight est à la fois le film le plus voté (3 250 069 votes) et le mieux noté de la sélection (9,1, la note maximale vue dans `describe()`).

> **Objectif** : trier les films par durée, pour voir les plus courts et surtout les plus longs, et décider quoi faire des durées exceptionnelles.

In [61]:
films.sort_values("runtimeMinutes")[["title", "startYear", "averageRating", "runtimeMinutes"]]

,title,startYear,averageRating,runtimeMinutes
3821,Winnie l'Ourson,2011,7.1,63
4237,Who Killed Captain Alex?,2010,7.7,64
847,Tetsuo,1989,6.9,67
2388,Interstella 5555,2003,7.8,68
823,Le petit dinosaure et la vallée des merveilles,1988,7.4,69
...,...,...,...,...
3514,La Ligue des Justiciers de Zack Snyder,2021,7.9,242
5633,Kill Bill: The Whole Bloody Affair,2004,8.7,253
1174,Gettysburg: la dernière bataille,1993,7.6,254
4331,Gangs of Wasseypur,2012,8.2,321


In [62]:
films.sort_values("runtimeMinutes")[["title", "startYear", "averageRating", "runtimeMinutes"]].tail(15)

,title,startYear,averageRating,runtimeMinutes
3955,The Bengal Files,2025,6.9,200
1835,Le Seigneur des anneaux : Le Retour du roi,2003,9.0,201
4096,Pushpa: The Rule - Part 2,2024,6.0,201
3701,Animal,2023,6.2,204
216,Le Message,1976,8.9,207
2691,Jodhaa Akbar,2008,7.5,213
5092,Dhurandhar,2025,8.2,214
355,La Porte du paradis,1980,6.7,219
5289,Dhurandhar: The Revenge,2026,8.2,235
3313,Love Exposure,2008,8.0,237


> **Constat** : les durées les plus longues sont réelles (Shoah : 566 minutes, Gangs of Wasseypur : 321 minutes…), ce ne sont pas des erreurs. Mais ces films sont exceptionnels : d'après le `describe()`, le premier quartile (ligne `25%`) est à 97 minutes et le troisième quartile (ligne `75%`) à 120 minutes, donc la moitié des films durent entre 97 et 120 minutes.

> **Décision** : on garde ces films, mais on plafonnera leur durée à 180 minutes (3 heures). Au-delà, « très long » suffit : Shoah n'a pas besoin d'être considéré comme trois fois plus long qu'un film de 3 heures. Sinon, ce seul film étirerait toute l'échelle des durées.

## 3. Une même échelle pour tous

> **Objectif** : comprendre, sur deux films qui se ressemblent (Alien et sa suite Aliens), pourquoi des colonnes d'échelles très différentes faussent le calcul de distance.

> **Objectif** : retrouver Alien et sa suite parmi les films dont le titre contient « Alien », pour récupérer leurs identifiants (`tconst`).

In [63]:
filtre_alien = films["title"].str.contains("Alien")
films[filtre_alien][["tconst", "title", "original_title", "startYear"]]

,tconst,title,original_title,startYear
294,tt0078748,"Alien, le 8ème passager",Alien,1979
646,tt0090605,"Aliens, le retour",Aliens,1986
1078,tt0103644,Alien 3,Alien³,1992
1508,tt0118583,"Alien, la résurrection",Alien Resurrection,1997
3017,tt0892782,Monstres contre Aliens,Monsters vs Aliens,2009
4265,tt18412256,Alien: Romulus,Untitled Alien Film,2024
4582,tt2316204,Alien: Covenant,Alien: Covenant,2017


> **Constat** : plusieurs films de la saga contiennent « Alien » dans leur titre. Les deux qui nous intéressent sont Alien, le 8ème passager (1979, `tt0078748`) et sa suite Aliens, le retour (1986, `tt0090605`).

In [64]:
deux_films = films[films["tconst"].isin(["tt0078748", "tt0090605"])]
deux_films[["title", "startYear", "runtimeMinutes", "averageRating", "numVotes"]]

,title,startYear,runtimeMinutes,averageRating,numVotes
294,"Alien, le 8ème passager",1979,117,8.4,1083357
646,"Aliens, le retour",1986,137,8.4,852599


In [65]:
numeriques[films["tconst"].isin(["tt0078748", "tt0090605"])]

,startYear,runtimeMinutes,averageRating,numVotes,log_votes
294,1979,117,8.4,1083357,6.034772
646,1986,137,8.4,852599,5.930745


> **Constat** : les deux films ont la même note (8,4) et des popularités proches (`log_votes` : 6,03 et 5,93). Ils diffèrent surtout par la durée (20 minutes d'écart) et par l'année (7 ans d'écart).

> **Objectif** : calculer la distance euclidienne entre Alien et Aliens sur les 4 colonnes retenues (année, durée, note, `log_votes`), en regardant la part de chaque colonne dans le résultat.

In [66]:
colonnes = ["startYear", "runtimeMinutes", "averageRating", "log_votes"]
les_deux_aliens = numeriques[films["tconst"].isin(["tt0078748", "tt0090605"])][colonnes]
ecarts = les_deux_aliens.iloc[0] - les_deux_aliens.iloc[1]
ecarts

startYear         -7.000000
runtimeMinutes   -20.000000
averageRating      0.000000
log_votes          0.104027
dtype: float64

In [67]:
ecarts ** 2

startYear          49.000000
runtimeMinutes    400.000000
averageRating       0.000000
log_votes           0.010822
dtype: float64

In [68]:
print(f"Distance euclidienne entre les deux films : {np.sqrt((ecarts ** 2).sum()):.2f}")

Distance euclidienne entre les deux films : 21.19


> **Constat** : la distance vaut 21,19. Sur les 449 de la somme des carrés, la durée en apporte 400 (environ 89 %) et l'année 49 ; la note et la popularité ne comptent quasiment pas. Avec les valeurs brutes, la distance mesure donc surtout un écart de durée. Un écart de 20 minutes pèse beaucoup plus qu'un écart de note de 1 point, alors que pour un spectateur, la note compte au moins autant.

> **Remarque** : prenons deux films identiques en tout, sauf l'année (10 ans d'écart) : la distance vaut 10. Prenons maintenant deux films identiques en tout, sauf la note (1 point d'écart) : la distance vaut 1. L'année pèse donc 10 fois plus. Pourtant, 1 point de note couvre près d'un tiers de l'échelle des notes (de 6 à 9,1), alors que 10 ans ne représentent que 18 % de la période (1970–2026). La distance brute dépend donc des unités de chaque colonne, pas de ce qui compte pour un spectateur : c'est un problème pour la recommandation.

> **Objectif** : ramener les 4 colonnes chiffrées entre 0 et 1 avec la normalisation min-max : chaque valeur x devient (x − min) / (max − min), avec le min et le max de sa colonne. Le plus petit devient 0, le plus grand 1, le reste se range entre les deux. On procède en deux étapes :
> 1. à la main, sur nos deux films seulement, pour comprendre la formule et voir son effet sur la distance ;
> 2. automatiquement, sur tout le catalogue, avec `MinMaxScaler` de scikit-learn.

> **Objectif de l'étape 1** : calculer le min et le max de chaque colonne sur tout le catalogue (avec le plafond de 180 minutes pour la durée), normaliser Alien et Aliens, puis recalculer leur distance.

In [69]:
mini = numeriques[colonnes].min()
maxi = numeriques[colonnes].max()
maxi["runtimeMinutes"] = 180   # plafond décidé
pd.DataFrame({"min": mini, "max": maxi})

,min,max
startYear,1970.000000,2026.000000
runtimeMinutes,63.000000,180.000000
averageRating,6.000000,9.100000
log_votes,4.000174,6.511893


In [70]:
les_deux_norm = (les_deux_aliens - mini) / (maxi - mini)
les_deux_norm

,startYear,runtimeMinutes,averageRating,log_votes
294,0.160714,0.461538,0.774194,0.810042
646,0.285714,0.632479,0.774194,0.768625


In [71]:
ecarts_norm = les_deux_norm.iloc[0] - les_deux_norm.iloc[1]
print(f"Distance après normalisation : {np.sqrt((ecarts_norm ** 2).sum()):.2f}")

Distance après normalisation : 0.22


> **Constat** : la distance vaut maintenant 0,22. On ne peut pas la comparer à 21,19 : les unités ont changé. Pour savoir si la normalisation a vraiment corrigé le problème, il faut regarder colonne par colonne.

> **Objectif** : mesurer, avant et après normalisation, la part de chaque colonne dans la distance, ainsi que la taille de chaque écart par rapport à l'étendue de sa colonne.

In [72]:
ecarts_norm.abs()

startYear         0.125000
runtimeMinutes    0.170940
averageRating     0.000000
log_votes         0.041417
dtype: float64

In [73]:
(ecarts_norm ** 2) / (ecarts_norm ** 2).sum()

startYear         0.335582
runtimeMinutes    0.627577
averageRating     0.000000
log_votes         0.036841
dtype: float64

In [74]:
(ecarts ** 2) / (ecarts ** 2).sum()

startYear         0.109129
runtimeMinutes    0.890847
averageRating     0.000000
log_votes         0.000024
dtype: float64

> **Constat** : la normalisation a réglé le problème des unités : chaque écart est mesuré par rapport à l'étendue de sa colonne (20 minutes représentent 17 % de l'étendue des durées, 7 ans 12,5 % de la période). Mais la durée apporte encore environ 63 % de la somme des carrés (contre 89 % avant normalisation), l'année 34 %, la popularité 4 % et la note 0 %.
>
> Ce n'est pas un défaut de calcul : ces deux films diffèrent réellement surtout par leur durée, et le carré amplifie le plus grand écart. Reste une vraie question, que la normalisation ne tranche pas : *faut-il que 20 minutes de durée comptent autant pour juger de la ressemblance entre deux films ?* Deux pistes pour la suite : l'ajout des genres, des mots-clés et de la langue diluera le poids de ces 4 colonnes ; et on pourra donner un poids plus faible aux colonnes chiffrées si elles comptent trop.

> **Objectif de l'étape 2** : appliquer la normalisation min-max à tout le catalogue avec `MinMaxScaler` de scikit-learn. D'abord sans plafond sur la durée, pour mesurer l'effet des films très longs.

In [75]:
from sklearn.preprocessing import MinMaxScaler

In [76]:
sans_plafond = MinMaxScaler().fit_transform(numeriques[colonnes])
sans_plafond = pd.DataFrame(sans_plafond, columns=colonnes)
sans_plafond.describe()

,startYear,runtimeMinutes,averageRating,log_votes
count,6089.000000,6089.000000,6089.000000,6089.000000
mean,0.642520,0.094380,0.271171,0.301764
std,0.249659,0.042022,0.189294,0.209221
min,0.000000,0.000000,0.000000,0.000000
25%,0.482143,0.067594,0.129032,0.122764
50%,0.696429,0.087475,0.258065,0.272373
75%,0.839286,0.113320,0.387097,0.452852
max,1.000000,1.000000,1.000000,1.000000


> **Constat** : sans plafond, la durée normalisée a une moyenne de 0,094, et les trois quarts des films sont en dessous de 0,113 (ligne `75%`). Shoah (566 minutes) occupe la valeur 1 et étire toute l'échelle : les films « normaux » sont écrasés près de 0, et un écart d'une demi-heure entre deux d'entre eux ne compte presque plus. Les trois autres colonnes sont bien mieux réparties entre 0 et 1.

> **Objectif** : appliquer la décision prise plus haut (durée plafonnée à 180 minutes), puis refaire la normalisation et comparer la durée normalisée. La méthode `clip(upper=180)` remplace toute valeur supérieure à 180 par 180, et laisse les autres inchangées.

In [77]:
numeriques["runtimeMinutes"] = numeriques["runtimeMinutes"].clip(upper=180)
numeriques["runtimeMinutes"].max()

np.int64(180)

In [78]:
numeriques_norm = MinMaxScaler().fit_transform(numeriques[colonnes])
numeriques_norm = pd.DataFrame(numeriques_norm, columns=colonnes)
numeriques_norm.describe()

,startYear,runtimeMinutes,averageRating,log_votes
count,6089.000000,6089.000000,6089.000000,6089.000000
mean,0.642520,0.403967,0.271171,0.301764
std,0.249659,0.167726,0.189294,0.209221
min,0.000000,0.000000,0.000000,0.000000
25%,0.482143,0.290598,0.129032,0.122764
50%,0.696429,0.376068,0.258065,0.272373
75%,0.839286,0.487179,0.387097,0.452852
max,1.000000,1.000000,1.000000,1.000000


> **Constat** : avec le plafond, la moyenne de la durée normalisée passe de 0,094 à 0,40. Les films « normaux » occupent maintenant une bonne partie de l'échelle de 0 à 1, au lieu d'être écrasés près de 0 : un écart d'une demi-heure entre deux films redevient visible. Les films de plus de 3 heures valent tous 1 (« très long »).

> **Remarque** : une autre solution aurait été de prendre le logarithme de la durée, comme pour les votes. On a préféré le plafond : il est plus simple à expliquer (« au-delà de 3 heures, c'est très long ») et il garde les durées ordinaires exactement proportionnelles.

> **Remarque (pour la suite)** : les genres sont codés en 0 ou 1, donc un écart sur un genre vaut au plus 1. Avec la normalisation, c'est aussi le cas pour chaque colonne chiffrée. Quand on réunira genres et colonnes chiffrées dans une même distance, aucune colonne n'écrasera les autres à cause de son unité. Sans normalisation, 20 minutes d'écart compteraient autant que 20 genres différents.

> **Objectif** : vérifier que deux colonnes chiffrées ne disent pas la même chose. Si deux colonnes sont très corrélées, l'information qu'elles portent compte deux fois dans la distance. La corrélation va de -1 à 1 : plus elle est proche de 1 (ou de -1), plus les deux colonnes varient ensemble.

In [79]:
numeriques_norm.corr().round(2)

,startYear,runtimeMinutes,averageRating,log_votes
startYear,1.00,0.15,-0.10,0.07
runtimeMinutes,0.15,1.00,0.31,0.19
averageRating,-0.10,0.31,1.00,0.29
log_votes,0.07,0.19,0.29,1.00


> **Constat** : la plus forte corrélation est de 0,31 (entre la durée et la note) ; toutes les autres sont inférieures à 0,3 en valeur absolue. Aucune colonne ne double une autre : on garde les quatre.

## 4. Les films voisins

> **Objectif** : réunir, pour chaque film, ses 21 colonnes de genres et ses 4 colonnes chiffrées normalisées en une seule ligne de 25 nombres. C'est sur ces lignes qu'on calculera les distances entre films.

In [80]:
X1 = pd.concat([genres, numeriques_norm], axis=1)
X1.shape

(6089, 25)

> **Constat** : `X1` a 6 089 lignes (une par film) et 25 colonnes (21 genres + 4 colonnes chiffrées), toutes comprises entre 0 et 1.

> **Objectif** : recalculer la distance entre Alien et Aliens, cette fois avec les genres, pour voir leur poids face aux colonnes chiffrées.

In [81]:
i_alien = films[films["tconst"] == "tt0078748"].index[0]
i_aliens = films[films["tconst"] == "tt0090605"].index[0]
ecarts_X = X1.iloc[i_alien] - X1.iloc[i_aliens]
# On n'affiche que les colonnes où les deux films diffèrent
ecarts_X[ecarts_X != 0]

Action           -1.000000
Adventure        -1.000000
Sci-Fi            1.000000
startYear        -0.125000
runtimeMinutes   -0.170940
log_votes         0.041417
dtype: float64

In [82]:
print(f"Distance avec les genres : {np.sqrt((ecarts_X ** 2).sum()):.2f}")

Distance avec les genres : 1.75


> **Constat** : Alien et Aliens diffèrent sur 3 genres (Sci-Fi d'un côté, Action et Adventure de l'autre). Ces 3 genres apportent 3 à la somme des carrés, contre environ 0,05 pour les 4 colonnes chiffrées : la distance (1,75) est maintenant presque entièrement due aux genres. C'est l'inverse du problème de départ : ce sont désormais les genres qui dominent. On pourra rééquilibrer plus tard en donnant un poids à chaque bloc de colonnes.

> **Objectif** : trouver les 5 films les plus proches d'Alien. On utilise l'algorithme des k plus proches voisins (KNN), dans sa version la plus simple : `NearestNeighbors` de scikit-learn se contente de trouver les voisins, sans rien prédire (ni classification comme `KNeighborsClassifier`, ni régression comme `KNeighborsRegressor`). C'est de l'apprentissage non supervisé. La distance utilisée est la distance euclidienne sur les 25 colonnes de `X1`. On demande 6 voisins, car le plus proche d'Alien… est Alien lui-même (distance 0).

In [83]:
from sklearn.neighbors import NearestNeighbors

In [84]:
modele1 = NearestNeighbors(n_neighbors=6)
modele1.fit(X1)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


> **Remarque** : dans scikit-learn, tous les outils ont une méthode `fit`, mais elle ne fait pas la même chose partout. Pour un modèle qui prédit (une régression linéaire, par exemple), `fit` *apprend* des paramètres à partir des données. Pour `NearestNeighbors`, il n'y a rien à apprendre : `fit` se contente de mémoriser les 6 089 lignes de `X1` et de les ranger pour accélérer la recherche (comme un annuaire classé par ordre alphabétique). Le vrai travail est fait plus tard, par `kneighbors`, qui calcule les distances au moment de la question. C'est pourquoi on dit que le KNN est un algorithme « paresseux » (*lazy learning*).

In [85]:
distances, indices = modele1.kneighbors(X1.iloc[[i_alien]])
indices, distances

(array([[ 294,  279, 2749,  302,  321,  512]]),
 array([[0.        , 0.54765093, 0.75622937, 0.80135405, 0.8325464 ,
         0.86381512]]))

> **Constat** : `kneighbors` renvoie deux tableaux : les positions des 6 films les plus proches (`indices`) et leurs distances (`distances`), du plus proche au plus éloigné. Le premier est Alien lui-même (position 294, distance 0). Les 5 autres sont ses voisins, à des distances de 0,55 à 0,86.

> **Objectif** : afficher les titres, années et genres des 6 films trouvés, avec leur distance à Alien.

In [86]:
voisins = films.iloc[indices[0]][["title", "startYear", "genres"]]
voisins["distance"] = distances[0]
voisins

,title,startYear,genres,distance
294,"Alien, le 8ème passager",1979,"Horror,Sci-Fi",0.000000
279,L'Invasion des profanateurs,1978,"Horror,Sci-Fi",0.547651
2749,28 semaines plus tard,2007,"Horror,Sci-Fi",0.756229
302,Chromosome 3,1979,"Horror,Sci-Fi",0.801354
321,Phantasm,1979,"Horror,Sci-Fi",0.832546
512,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.863815


> **Constat** : les 5 voisins d'Alien ont exactement les mêmes genres (Horror, Sci-Fi) et sont presque tous sortis entre 1978 et 1983. Les suites d'Alien n'apparaissent pas : Aliens diffère d'Alien sur 3 genres, donc sa distance vaut au moins √3 ≈ 1,73, bien plus que ces voisins (0,55 à 0,86). Le moteur juge sur l'étiquette (genres, époque), pas sur le contenu : il ignore que les films de la saga racontent la même histoire. Il manque une description de ce dont parle le film : ce sera le rôle des mots-clés.

> **Objectif** : écrire des fonctions pour tester facilement le moteur sur n'importe quel film. `chercher(mot)` affiche les films dont le titre contient un mot, avec leur identifiant `tconst`. `recommander(tconst, X, modele)` affiche les 5 films les plus proches d'un film donné, pour une matrice `X` et un modèle donnés ; `recommander1(tconst)` l'applique à la version 1 (`X1` et `modele1`).

In [87]:
def chercher(mot):
    filtre = films["title"].str.contains(mot, case=False)
    return films[filtre][["tconst", "title", "startYear", "genres"]]

In [88]:
chercher("taxi")

,tconst,title,startYear,genres
1787,tt0152930,Taxi,1998,"Action,Comedy,Crime"
1873,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2547,tt0410730,Taxidermie,2006,"Comedy,Horror"
2795,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
5363,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
5756,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [89]:
chercher("alien")

,tconst,title,startYear,genres
294,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
646,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1053,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1078,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1508,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2215,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
3017,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4265,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"
4582,tt2316204,Alien: Covenant,2017,"Horror,Sci-Fi,Thriller"


In [90]:
chercher("matrix")

,tconst,title,startYear,genres
1728,tt0133093,Matrix,1999,"Action,Sci-Fi"
1970,tt0234215,Matrix Reloaded,2003,"Action,Sci-Fi"
1987,tt0242653,Matrix Revolutions,2003,"Action,Sci-Fi"


> **Remarque** : *Matrix Resurrections* (2021), quatrième film de la saga, n'est pas dans le catalogue. Sa note IMDb est de 5,6/10 (environ 293 000 votes, source : imdb.com/title/tt10838180), sous le seuil de 6 retenu par le filtre de l'équipe. Ce seuil écarte donc aussi des suites très connues : un choix à garder en tête, et éventuellement à discuter.

> **Idée (à discuter en équipe)** : compléter les sagas. Si un film d'une saga est dans la sélection, on pourrait ajouter les autres films de la même saga, avec des garde-fous :
> - un seuil de note plus bas (par exemple 5 au lieu de 6), pour récupérer des suites comme *Matrix Resurrections* sans importer les suites vraiment ratées ;
> - un nombre de votes suffisant, pour écarter les suites obscures ;
> - une limite pour les très longues sagas (Star Trek, James Bond…).
>
> Point à vérifier : quelle source indique qu'un film appartient à une saga (ni IMDb ni les colonnes chargées depuis TMDB ne le donnent directement).
>
> Piste simple : les sagas « cultes » sont peu nombreuses (quelques dizaines au plus). On peut les définir à la main dans un petit fichier `sagas.csv` (colonnes `saga` et `tconst`). On décide alors film par film, et la liste pourrait aussi servir dans l'appli, pour afficher « Dans la même saga ».

> **Remarque** : `recommander` reçoit la matrice et le modèle en paramètres, au lieu d'aller chercher des variables du notebook. On pourra ainsi interroger plusieurs versions du moteur (`X1` et `modele1`, puis `X2` et `modele2`…) et les comparer côte à côte. Passer `X` en paramètre ne le copie pas : la fonction reçoit une référence vers le même tableau.

In [91]:
def recommander(tconst, X, modele):
    i = films[films["tconst"] == tconst].index[0]
    distances, indices = modele.kneighbors(X.iloc[[i]])
    voisins = films.iloc[indices[0][1:]][["tconst", "title", "startYear", "genres"]]
    voisins["distance"] = distances[0][1:]
    return voisins

In [92]:
def recommander1(tconst):
    return recommander(tconst, X1, modele1)

In [93]:
recommander1("tt0078748")

,tconst,title,startYear,genres,distance
279,tt0077745,L'Invasion des profanateurs,1978,"Horror,Sci-Fi",0.547651
2749,tt0463854,28 semaines plus tard,2007,"Horror,Sci-Fi",0.756229
302,tt0078908,Chromosome 3,1979,"Horror,Sci-Fi",0.801354
321,tt0079714,Phantasm,1979,"Horror,Sci-Fi",0.832546
512,tt0086491,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.863815


In [94]:
recommander1("tt0077745")

,tconst,title,startYear,genres,distance
302,tt0078908,Chromosome 3,1979,"Horror,Sci-Fi",0.306644
321,tt0079714,Phantasm,1979,"Horror,Sci-Fi",0.356728
512,tt0086491,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.372510
660,tt0091083,From Beyond : Aux portes de l'au-delà,1986,"Horror,Sci-Fi",0.416171
246,tt0076590,Rage,1977,"Horror,Sci-Fi",0.463361


In [95]:
recommander1("tt0078908")

,tconst,title,startYear,genres,distance
321,tt0079714,Phantasm,1979,"Horror,Sci-Fi",0.074865
660,tt0091083,From Beyond : Aux portes de l'au-delà,1986,"Horror,Sci-Fi",0.153209
512,tt0086491,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.167692
246,tt0076590,Rage,1977,"Horror,Sci-Fi",0.184944
184,tt0073705,Frissons,1975,"Horror,Sci-Fi",0.190999


> **Constat** : en enchaînant les recommandations (Alien → L'Invasion des profanateurs → Chromosome 3), on reste dans un petit groupe de films Horror, Sci-Fi de 1975–1986, et les distances diminuent (de 0,55–0,86 à 0,07–0,19) : on s'enfonce dans un groupe de films très proches les uns des autres, qui se recommandent entre eux. Ce moteur tourne un peu en rond, moins toutefois qu'un moteur plus précis : précision et diversité tirent en sens contraire. Ce point sera traité dans l'application (exclure les films déjà vus, mélanger proches et plus lointains).

In [96]:
chercher("taxi")

,tconst,title,startYear,genres
1787,tt0152930,Taxi,1998,"Action,Comedy,Crime"
1873,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2547,tt0410730,Taxidermie,2006,"Comedy,Horror"
2795,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
5363,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
5756,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [97]:
recommander1("tt0152930")

,tconst,title,startYear,genres,distance
1873,tt0183869,Taxi 2,2000,"Action,Comedy,Crime",0.181644
1555,tt0119229,Tueurs à gages,1997,"Action,Comedy,Crime",0.183349
1037,tt0102266,Le Dernier Samaritain,1991,"Action,Comedy,Crime",0.189786
2115,tt0281364,La petite moutarde qui monte au nez,2001,"Action,Comedy,Crime",0.193391
1861,tt0181316,Flic de haut vol,1999,"Action,Comedy,Crime",0.197408


> **Constat** : pour Taxi, Taxi 2 arrive en tête, suivi de comédies d'action policières de la même époque (1991–2001), toutes Action, Comedy, Crime. Le résultat est correct, car ces genres sont précis et partagés par la suite. Mais les distances sont très proches (0,18 à 0,20) : le moteur départage des films au même profil sur de petits écarts d'année ou de note. Il ignore ce qui fait l'identité de Taxi (film français, Marseille, voitures) : les voisins proposés sont américains.

> **Remarque** : ce premier moteur (21 genres et 4 colonnes chiffrées, distance euclidienne, sans poids) est appelé **version 1** dans la suite (`X1`, `modele1`, `recommander1`). La partie 5 construit une version 2, qui ajoute les mots-clés, des poids et la distance cosinus.

## 5. Le contenu des films : les mots-clés

> **Remarque** : dans cette partie, les cellules **Article** signalent les valeurs et les remarques reprises dans l'article « TF-IDF et similarité cosinus », qui accompagne le développement du moteur.

> **Objectif** : ajouter au moteur une description du contenu des films, grâce aux mots-clés TMDB (colonne `keywords`). On commence par regarder à quoi ressemblent ces mots-clés pour Alien et Aliens.

In [98]:
chercher("alien")

,tconst,title,startYear,genres
294,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
646,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1053,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1078,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1508,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2215,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
3017,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4265,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"
4582,tt2316204,Alien: Covenant,2017,"Horror,Sci-Fi,Thriller"


In [99]:
filtre_deux = films["tconst"].isin(["tt0078748", "tt0090605"])
for texte in films[filtre_deux]["keywords"]:
    print(texte)
    print()

android, spacecraft, space marine, biology, dystopia, countdown, space suit, beheading, space travel, cowardice, alien, space, female protagonist, parasite, space opera, cosmos, xenomorph

android, space marine, extraterrestrial technology, spaceman, space travel, settler, colony, cryogenics, vacuum, space colony, warrior woman, alien, space, female protagonist, creature, desolate, female hero, xenomorph, desolate planet, strong female protagonist



In [100]:
mots_alien = films[films["tconst"] == "tt0078748"]["keywords"].iloc[0].split(", ")
mots_aliens = films[films["tconst"] == "tt0090605"]["keywords"].iloc[0].split(", ")
len(mots_alien), len(mots_aliens)

(17, 20)

In [101]:
communs = []
for mot in mots_alien:
    if mot in mots_aliens:
        communs.append(mot)
print("Nombre de mots clés commun :", len(communs))
communs

Nombre de mots clés commun : 7


['android',
 'space marine',
 'space travel',
 'alien',
 'space',
 'female protagonist',
 'xenomorph']

> **Constat** : Alien a 17 mots-clés, Aliens 20, et ils en partagent 7 : android, space marine, space travel, alien, space, female protagonist, xenomorph. Ces mots décrivent le contenu commun des deux films (une créature extraterrestre, l'espace, une héroïne), ce que les genres ne disaient pas.

> **Article** : section 1.1 (Alien et Aliens partagent sept mots-clés) et section 5.2 (les synonymes : « spacecraft » figure parmi les mots-clés d'Alien).

> **Objectif** : compter, pour chacun des 7 mots-clés communs, combien de films du catalogue le possèdent, pour voir s'ils sont rares ou fréquents.

In [102]:
listes_mots = films["keywords"].str.split(", ")

for mot in communs:
    nb = 0
    for liste in listes_mots:
        if mot in liste:
            nb = nb + 1
    print(mot, ":", nb, "films")

android : 37 films
space marine : 21 films
space travel : 37 films
alien : 98 films
space : 41 films
female protagonist : 62 films
xenomorph : 5 films


> **Constat** : les 7 mots communs n'ont pas la même rareté. « xenomorph » n'apparaît que dans 5 films du catalogue (sans doute la saga Alien), « space marine » dans 21, alors que « alien » apparaît dans 98 films et « female protagonist » dans 62. Partager un mot rare en dit beaucoup plus sur la ressemblance de deux films que partager un mot fréquent.

> **Article** : section 4.5 (colonne df du tableau des 7 mots communs).

In [103]:
for i in range(len(films)):
    if "xenomorph" in listes_mots.iloc[i]:
        print(films["title"].iloc[i], films["startYear"].iloc[i])

Alien, le 8ème passager 1979
Aliens, le retour 1986
Alien 3 1992
Alien, la résurrection 1997
Alien: Covenant 2017


> **Constat** : les 5 films qui ont le mot-clé « xenomorph » sont bien des films de la saga Alien (1979, 1986, 1992, 1997, 2017). Ce mot rare suffit presque à lui seul à identifier la saga. Mais Alien: Romulus (2024), présent dans le catalogue, n'apparaît pas : regardons ses mots-clés.

In [104]:
chercher("romulus")

,tconst,title,startYear,genres
4265,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"


In [105]:
films[films["tconst"] == "tt18412256"]["keywords"]

4265    alien, space
Name: keywords, dtype: str

> **Constat** : TMDB ne donne que deux mots-clés à Alien: Romulus (« alien, space »). Ce film récent était encore peu décrit par la communauté au moment de l'extraction. Avec si peu de mots-clés, il sera difficile au moteur de le rapprocher du reste de la saga.

> **Article** : section 5.2 (Ce que TF-IDF ne voit pas, la qualité des descriptions : Alien: Romulus n'a que deux mots-clés, « alien » et « space »).

### Le vocabulaire

> **Objectif** : compter les films et les mots-clés différents de tout le catalogue (la taille du vocabulaire). Le nombre de mots-clés différents est la dimension p de l'espace dans lequel chaque film deviendra un vecteur.

In [106]:
vocabulaire = set()
for liste in listes_mots:
    for mot in liste:
        vocabulaire.add(mot)

print("films :", len(films))
print("mots-clés différents :", len(vocabulaire))

films : 6089
mots-clés différents : 12709


> **Objectif** : compter les mots-clés de chaque film, pour savoir combien de ces coordonnées un film utilise réellement.

In [107]:
nb_mots = []
for liste in listes_mots:
    nb_mots.append(len(liste))
nb_mots = pd.Series(nb_mots)

print("minimum :", nb_mots.min())
print("moyenne :", round(nb_mots.mean(), 1))
print("maximum :", nb_mots.max())

minimum : 1
moyenne : 9.9
maximum : 75


> **Constat** : le catalogue compte n = 6 089 films, qui utilisent 12 709 mots-clés différents. Chaque film deviendra donc un vecteur de dimension p = 12 709, alors qu'il n'a qu'une dizaine de mots-clés en moyenne, et 75 au plus. Ces vecteurs sont presque entièrement remplis de 0 : on dit que la matrice des films est *creuse*.

> **Article** : section 2.4 (Dimension et creux) : n = 6 089 films, p = 12 709 mots-clés, et un film n'a que « quelques dizaines de mots-clés au plus ».

### La fréquence documentaire

> **Objectif** : calculer, pour chaque mot-clé, le nombre de films qui le contiennent : c'est sa *fréquence documentaire*, notée df. On parcourt `set(liste)` plutôt que `liste` : un `set` supprime les doublons, et un film ne doit compter qu'une fois par mot, même si ce mot est répété dans sa liste (voir la partie suivante).

In [108]:
compte = {}
for liste in listes_mots:
    for mot in set(liste):    # set : un film ne compte qu'une fois par mot
        compte[mot] = compte.get(mot, 0) + 1

print("xenomorph :", compte["xenomorph"])
print("alien :", compte["alien"])

xenomorph : 5
alien : 98


> **Constat** : on retrouve les valeurs obtenues plus haut avec une boucle mot par mot (5 films pour « xenomorph », 98 pour « alien ») : `compte[mot]` est bien la fréquence documentaire du mot.

> **Objectif** : compter les mots-clés présents dans un seul film (df = 1). Un tel mot ne peut rapprocher aucun film d'un autre.

In [109]:
nb_uniques = 0
for mot in compte:
    if compte[mot] == 1:
        nb_uniques = nb_uniques + 1

print("mots présents dans un seul film :", nb_uniques)
print("part du vocabulaire (%) :", round(nb_uniques / len(vocabulaire) * 100, 1))
print("mots restants :", len(vocabulaire) - nb_uniques)

mots présents dans un seul film : 6233
part du vocabulaire (%) : 49.0
mots restants : 6476


> **Constat** : 6 233 mots-clés, soit 49 % du vocabulaire, n'apparaissent que dans un seul film. Dans le produit scalaire de deux films, un mot ne compte que si les deux films l'ont : ces mots n'ajoutent donc rien à la ressemblance entre films. Pire, ils allongent le vecteur de leur film, ce qui l'éloigne un peu de tous les autres. En les retirant, il reste 6 476 mots-clés utiles.
>
> **Décision** : écarter les mots présents dans un seul film. Avec scikit-learn, c'est l'option `min_df=2` (voir plus bas).

> **Article** : section 2.4 (6 233 mots uniques, 49 %, 6 476 mots restants) et section 4.2 (remarque 1 : le cas df = 1).

### Les mots-clés répétés

> **Objectif** : compter les films dont la liste contient un mot-clé répété. Un `set` supprime les doublons : s'il est plus court que la liste, c'est qu'un mot y est répété.

In [110]:
nb_films = 0
for liste in listes_mots:
    if len(set(liste)) < len(liste):
        nb_films = nb_films + 1
nb_films

153

> **Objectif** : voir quels mots-clés sont le plus souvent répétés, et dans combien de films.

In [111]:
repetes = {}
for liste in listes_mots:
    for mot in set(liste):
        if liste.count(mot) > 1:
            repetes[mot] = repetes.get(mot, 0) + 1
pd.Series(repetes).sort_values(ascending=False).head(5)

new york city    26
england          20
california       12
france           10
japan             9
dtype: int64

> **Objectif** : afficher les films dans lesquels « england » est répété, avec le nombre de répétitions.

In [112]:
for i in range(len(films)):
    nb = listes_mots.iloc[i].count("england")
    if nb > 1:
        print(films["title"].iloc[i], ":", nb, "fois")

Les Chiens de paille : 2 fois
Vivre et laisser mourir : 2 fois
L'homme au pistolet d'or : 2 fois
Rien que pour vos yeux : 2 fois
Dangereusement vôtre : 2 fois
Tuer n'est pas jouer : 2 fois
Demain ne meurt jamais : 2 fois
Coup de foudre à Notting Hill : 2 fois
Le Journal de Bridget Jones : 2 fois
L'escorte : 2 fois
Miss Pettigrew : 2 fois
Auprès de moi toujours : 2 fois
RED 2 : 2 fois
Il était temps : 2 fois
Conjuring 2 : Le Cas Enfield : 2 fois
Le BGG : Le Bon Gros Géant : 2 fois
The Lady in the Van : 2 fois
Le Merveilleux Jardin secret de Bella Brown : 2 fois
6 Days : 2 fois
Juliet, Naked : 2 fois


> **Objectif** : chercher les mots-clés écrits plus de deux fois dans la liste d'un même film.

In [113]:
for i in range(len(films)):
    liste = listes_mots.iloc[i]
    for mot in set(liste):
        if liste.count(mot) > 2:
            print(films["title"].iloc[i], ":", mot, liste.count(mot), "fois")

Il était une fois... : new york city 3 fois
Son of a Gun : australia 3 fois
Ocho apellidos vascos : spain 3 fois
Un jour de pluie à New York : new york city 3 fois


> **Objectif** : chercher les mots-clés présents dans un seul film, mais répétés dans sa liste. Ce sont eux qu'une boucle sans `set` compterait mal.

In [114]:
for i in range(len(films)):
    liste = listes_mots.iloc[i]
    for mot in set(liste):
        if compte[mot] == 1 and liste.count(mot) > 1:
            print(mot, ":", films["title"].iloc[i])

louisville : Le retour des morts-vivants
bali : Ticket to Paradise


> **Constat** : 153 films ont au moins un mot-clé répété dans leur liste TMDB. Les mots les plus souvent répétés sont des lieux : « new york city » est répété dans 26 films, « england » dans 20 (à chaque fois exactement deux fois). Seuls 4 films ont un mot écrit trois fois, et aucun n'en a un écrit davantage. Ces répétitions sont des erreurs de saisie : elles ne disent rien de plus sur le film.
>
> **Remarque** : deux de ces mots, « louisville » et « bali », n'apparaissent que dans un seul film. Sans le `set`, leur compte vaudrait 2 et ils échapperaient au décompte des mots uniques : c'est pourquoi la boucle `compte` parcourt `set(liste)`.

> **Remarque** : les autres mots répétés sont partagés par de nombreux films. Par défaut, `TfidfVectorizer` compte les occurrences : la répétition doublerait le poids de ces mots dans les films concernés, et ces films paraîtraient artificiellement plus proches des autres films new-yorkais ou anglais.
>
> **Décision** : lors du calcul TF-IDF, utiliser l'option `binary=True` : un mot-clé présent compte 1, qu'il soit répété ou non.

> **Article** : annexe B, section B.2 (codage du tf : 153 films ont un mot-clé répété, par exemple « england » écrit deux fois).

### L'IDF : le poids de la rareté

> **Objectif** : calculer le poids IDF des 7 mots-clés communs à Alien et Aliens, avec la formule de l'article : idf(t) = ln(n / df(t)) + 1, où n est le nombre de films et df(t) le nombre de films qui contiennent le mot t. Plus un mot est rare, plus son poids est grand.

In [115]:
n = len(films)

def idf(mot):
    return np.log(n / compte[mot]) + 1

for mot in communs:
    print(f"{mot} : {compte[mot]} films, idf = {idf(mot):.2f}")

android : 37 films, idf = 6.10
space marine : 21 films, idf = 6.67
space travel : 37 films, idf = 6.10
alien : 98 films, idf = 5.13
space : 41 films, idf = 6.00
female protagonist : 62 films, idf = 5.59
xenomorph : 5 films, idf = 8.10


> **Objectif** : comparer le mot le plus rare (« xenomorph ») et le plus fréquent (« alien ») : combien de fois plus rare, combien de fois plus lourd ?

In [116]:
print("rapport des df :", round(compte["alien"] / compte["xenomorph"], 1))
print("rapport des idf :", round(idf("xenomorph") / idf("alien"), 2))

rapport des df : 19.6
rapport des idf : 1.58


> **Constat** : « xenomorph » (5 films) est près de vingt fois plus rare que « alien » (98 films), mais son poids n'est que 1,58 fois plus grand (8,10 contre 5,13). C'est l'effet du logarithme : il favorise les mots rares sans écraser les autres.

> **Article** : section 4.2 (courbe de l'IDF, avec les points de « xenomorph », « space marine », « android » et « alien ») et section 4.5 (tableau des 7 mots communs, rapport 8,10 / 5,13 ≈ 1,58).

### Un exemple jouet

> **Objectif** : sur un petit exemple de 6 films (A à F), voir comment la pondération TF-IDF modifie les ressemblances. Les mots-clés sont inventés pour que chaque valeur se vérifie à la main ; A joue le rôle d'Alien.

In [117]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def decouper(texte):
    return texte.split(", ")    # un mot-clé = une expression entre virgules

jouet = ["xenomorph, spaceship, murder",    # A
         "xenomorph, android",              # B
         "murder, detective",               # C
         "murder, spaceship, princess",     # D
         "spaceship, android",              # E
         "murder, android"]                 # F

> **Objectif** : coder les 6 films en binaire (1 si le mot est présent, 0 sinon), puis calculer la similarité cosinus de A avec chaque film.

In [118]:
# Sans idf ni normalisation : la matrice des 0 et des 1
binaire = TfidfVectorizer(tokenizer=decouper, token_pattern=None,
                          binary=True, use_idf=False, norm=None)
X_jouet = binaire.fit_transform(jouet)
print(binaire.get_feature_names_out())
print(X_jouet.toarray().astype(int))

['android' 'detective' 'murder' 'princess' 'spaceship' 'xenomorph']
[[0 0 1 0 1 1]
 [1 0 0 0 0 1]
 [0 1 1 0 0 0]
 [0 0 1 1 1 0]
 [1 0 0 0 1 0]
 [1 0 1 0 0 0]]


In [119]:
# Cette matrice documents-termes peut-être affiché comme un DataFrame
pd.DataFrame(X_jouet.toarray(), columns=binaire.get_feature_names_out())

,android,detective,murder,princess,spaceship,xenomorph
0,0.0,0.0,1.0,0.0,1.0,1.0
1,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,1.0,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,1.0,1.0,0.0
4,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,1.0,0.0,0.0,0.0


In [120]:
cosine_similarity(X_jouet)[0].round(3)    # A, B, C, D, E, F

array([1.   , 0.408, 0.408, 0.667, 0.408, 0.408])

> **Constat** : en codage binaire, D est le plus proche de A (0,667) : ils partagent deux mots (murder, spaceship). Les quatre autres films partagent un seul mot avec A et sont à égalité (0,408), quel que soit ce mot.

> **Objectif** : refaire le calcul avec la pondération TF-IDF de l'article : idf non lissé (`smooth_idf=False`) et vecteurs non normalisés (`norm=None`).

In [121]:
tfidf_jouet = TfidfVectorizer(tokenizer=decouper, token_pattern=None,
                              smooth_idf=False, norm=None)
W_jouet = tfidf_jouet.fit_transform(jouet)
print(tfidf_jouet.get_feature_names_out())
print(tfidf_jouet.idf_.round(3))

['android' 'detective' 'murder' 'princess' 'spaceship' 'xenomorph']
[1.693 2.792 1.405 2.792 1.693 2.099]


In [122]:
cosine_similarity(W_jouet)[0].round(3)    # A, B, C, D, E, F

array([1.   , 0.537, 0.208, 0.448, 0.394, 0.295])

> **Constat** : le classement change complètement : B (0,537), D (0,448), E (0,394), F (0,295), C (0,208). Trois effets l'expliquent :
> - un mot partagé compte d'autant plus qu'il est rare : B (xenomorph, 2 films) passe devant F (murder, 4 films) ;
> - un mot qu'un film possède en propre l'éloigne, d'autant plus qu'il est rare : C et F partagent le même mot avec A, mais C a en plus « detective » (1 film) et F « android » (3 films) ;
> - un seul mot rare peut l'emporter sur deux mots courants : B passe devant D.
>
> En résumé, le codage binaire *compte* les mots partagés, la pondération TF-IDF les *pèse*.

> **Objectif** : comparer avec les réglages par défaut de scikit-learn : idf lissé, ln((1 + n) / (1 + df)) + 1, et vecteurs ramenés à une norme de 1.

In [123]:
defaut = TfidfVectorizer(tokenizer=decouper, token_pattern=None)
W_defaut = defaut.fit_transform(jouet)
print(defaut.idf_.round(3))
print(cosine_similarity(W_defaut)[0].round(3))    # A, B, C, D, E, F

[1.56  2.253 1.336 2.253 1.56  1.847]
[1.    0.511 0.247 0.501 0.399 0.315]


> **Constat** : le lissage fait comme si l'on ajoutait un film fictif contenant tous les mots. Les poids baissent, surtout ceux des mots rares (xenomorph passe de 2,099 à 1,847). Le classement reste le même (B, D, E, F, C), mais l'avance de B sur D se réduit (0,511 contre 0,501). La normalisation, elle, ne change aucun cosinus.

> **Article** : section 2.3 (la matrice des 6 films), section 3.3 (cosinus en codage binaire), section 4.4 (idf, cosinus TF-IDF et les trois constats) et annexe B, section B.2 (tableau sans lissage / avec lissage).

### TF-IDF sur le catalogue

> **Objectif** : appliquer TF-IDF aux mots-clés du catalogue, avec les deux décisions prises plus haut : `binary=True` (mots répétés) et `min_df=2` (mots présents dans un seul film). Les autres réglages restent ceux par défaut.

In [124]:
tfidf = TfidfVectorizer(tokenizer=decouper, token_pattern=None,
                        binary=True, min_df=2)
mots = tfidf.fit_transform(films["keywords"])
mots.shape

(6089, 6476)

> **Objectif** : compter les coefficients non nuls de la matrice. `mots` est une matrice creuse (*sparse*) : au lieu de stocker toutes ses cases, elle ne stocke que les cases non nulles, avec leur position. Leur nombre est donné par l'attribut `nnz` (*number of non-zeros*, « nombre de non-zéros »).

In [125]:
nb_cases = mots.shape[0] * mots.shape[1]
print("coefficients non nuls :", mots.nnz)
print("part des coefficients (%) :", round(mots.nnz / nb_cases * 100, 2))

coefficients non nuls : 54099
part des coefficients (%) : 0.14


> **Constat** : la matrice a 6 089 lignes (les films) et 6 476 colonnes : on retrouve exactement le nombre de mots restants calculé à la main. Elle ne contient que 54 099 coefficients non nuls, soit 0,14 % de ses cases.

> **Objectif** : comparer les poids IDF de scikit-learn (lissés) à ceux calculés à la main, pour les 7 mots communs à Alien et Aliens.

In [126]:
for mot in communs:
    j = tfidf.vocabulary_[mot]    # numéro de la colonne du mot
    print(f"{mot} : à la main {idf(mot):.2f}, scikit-learn {tfidf.idf_[j]:.2f}")

android : à la main 6.10, scikit-learn 6.08
space marine : à la main 6.67, scikit-learn 6.62
space travel : à la main 6.10, scikit-learn 6.08
alien : à la main 5.13, scikit-learn 5.12
space : à la main 6.00, scikit-learn 5.98
female protagonist : à la main 5.59, scikit-learn 5.57
xenomorph : à la main 8.10, scikit-learn 7.92


> **Constat** : sur le catalogue, le lissage ne change presque rien : les écarts sont de quelques centièmes, sauf pour « xenomorph » (8,10 contre 7,92), le plus rare. On peut donc garder le réglage par défaut.

> **Objectif** : voir pourquoi il faut donner à `TfidfVectorizer` notre propre fonction de découpage. Sans elle, scikit-learn découpe le texte en mots isolés.

In [127]:
mots_isoles = TfidfVectorizer()
mots_isoles.fit(films["keywords"])
print("space marine :", "space marine" in mots_isoles.vocabulary_)
print("marine :", "marine" in mots_isoles.vocabulary_)

space marine : False
marine : True


> **Constat** : avec le découpage par défaut, « space marine » n'existe plus comme terme : il est coupé en « space » et « marine », et l'expression perd son sens. La fonction `decouper` garde chaque mot-clé entier.

> **Remarque** : on ne nettoie pas le texte : pas de suppression des mots vides, pas de regroupement des formes d'un même mot (stemming ou lemmatisation). Ces traitements servent pour du texte libre, comme des avis ou des résumés. Nos mots-clés sont des étiquettes courtes, écrites en minuscules (« space marine », « xenomorph ») : le découpage aux virgules suffit.

> **Idée (à discuter en équipe)** : ajouter les résumés des films (colonne `overview` du fichier TMDB, absente du catalogue actuel) comme huitième bloc TF-IDF, avec son propre poids. Contrairement aux mots-clés, c'est du texte libre, en anglais : il faudrait alors le nettoyer (mots vides avec `stop_words="english"`, éventuellement lemmatisation ou stemming, en mesurant si ça aide) et régler `min_df`, `max_df`, voire `ngram_range=(1, 2)`. Atout : les résumés citent les personnages (Ripley, Neo, Astérix…), des mots rares qui relieraient les films d'une même saga. Coût : un texte plus bruité, et de nouveaux tests sur les films tests.

> **Article** : annexe B, section B.1 (code, taille de la matrice, 54 099 coefficients non nuls, 0,14 %) et section B.2 (découpage, `binary`, lissage, `min_df`).

### Modèle version 2

> **Objectif** : construire une nouvelle matrice `X2` en six blocs, chacun multiplié par son propre poids : les genres, chacune des 4 valeurs numériques normalisées (année, durée, note, popularité), et les mots-clés TF-IDF. On commence par les valeurs numériques : leurs 4 poids sont rangés dans un dictionnaire (0,7 pour toutes, pour commencer).

In [128]:
poids_num = {"startYear": 0.7, "runtimeMinutes": 0.7,
             "averageRating": 0.7, "log_votes": 0.7}

num_ponderes = numeriques_norm.copy()
for col in poids_num:
    num_ponderes[col] = numeriques_norm[col] * poids_num[col]
num_ponderes.head()

,startYear,runtimeMinutes,averageRating,log_votes
0,0.3875,0.329060,0.090323,0.272652
1,0.0000,0.179487,0.293548,0.007858
2,0.0000,0.221368,0.248387,0.022602
3,0.0000,0.317094,0.225806,0.146296
4,0.0000,0.197436,0.248387,0.123560


In [129]:
num_ponderes.describe()

,startYear,runtimeMinutes,averageRating,log_votes
count,6089.000000,6089.000000,6089.000000,6089.000000
mean,0.449764,0.282777,0.189820,0.211235
std,0.174762,0.117408,0.132506,0.146455
min,0.000000,0.000000,0.000000,0.000000
25%,0.337500,0.203419,0.090323,0.085935
50%,0.487500,0.263248,0.180645,0.190661
75%,0.587500,0.341026,0.270968,0.316996
max,0.700000,0.700000,0.700000,0.700000


> **Constat** : chaque colonne numérique va maintenant de 0 à 0,7 au lieu de 0 à 1. Les valeurs sont multipliées par le même nombre, donc l'ordre des films ne change pas dans chaque colonne ; seul change le poids de la colonne dans les distances.

> **Objectif** : assembler la matrice `X2` des six blocs : les genres multipliés par 0,7, les 4 colonnes numériques pondérées, et les mots-clés TF-IDF (poids 1). `mots` est une matrice creuse : `toarray()` la transforme en tableau ordinaire pour pouvoir la coller aux autres blocs.

In [130]:
X2 = np.hstack([0.7 * genres.values,
                num_ponderes.values,
                mots.toarray()])
X2 = pd.DataFrame(X2)
X2.shape

(6089, 6501)

> **Constat** : `X2` a 6 089 lignes et 6 501 colonnes : 21 genres, 4 colonnes numériques et 6 476 mots-clés.

> **Objectif** : vérifier qu'aucun bloc n'écrase les autres. Le bloc mots-clés a 6 476 colonnes, contre 21 pour les genres et 4 pour les valeurs numériques : pèse-t-il beaucoup plus lourd dans la distance ? On calcule, pour chaque film, la longueur (la norme) de chaque bloc, c'est-à-dire la racine de la somme des carrés de ses valeurs.

In [131]:
blocs = {"genres": X2.iloc[:, :21],
         "numériques": X2.iloc[:, 21:25],
         "mots-clés": X2.iloc[:, 25:]}

longueurs = {}
for nom in blocs:
    longueurs[nom] = np.sqrt((blocs[nom] ** 2).sum(axis=1))
longueurs = pd.DataFrame(longueurs)
longueurs.describe().round(2)

,genres,numériques,mots-clés
count,6089.00,6089.00,6089.00
mean,1.14,0.65,1.00
std,0.13,0.17,0.07
min,0.70,0.12,0.00
25%,0.99,0.53,1.00
50%,1.21,0.65,1.00
75%,1.21,0.76,1.00
max,1.21,1.24,1.00


> **Constat** : malgré ses milliers de colonnes, le bloc mots-clés a une longueur de 1 pour presque tous les films (ligne `25%` : déjà 1) : `TfidfVectorizer` divise chaque ligne par sa norme (réglage `norm="l2"`, par défaut). Les genres ont une longueur de 1,14 en moyenne (0,7 multiplié par la racine du nombre de genres du film), les valeurs numériques de 0,65. Les trois blocs sont du même ordre de grandeur : aucun n'écrase les autres, et ce sont les poids qui règlent leur importance.
>
> **Remarque** : le cours sur la similarité conseille de chercher les voisins sur le texte seul, car des milliers de colonnes TF-IDF « pèseraient presque toute la distance ». Ce n'est pas le cas ici, grâce à la normalisation de chaque ligne : le nombre de colonnes d'un bloc ne compte pas, seule sa longueur compte.

> **Objectif** : la ligne `min` montre qu'au moins un film a un bloc mots-clés de longueur 0. Compter ces films.

In [132]:
(longueurs["mots-clés"] == 0).sum()

np.int64(27)

> **Constat** : 27 films ont un bloc mots-clés entièrement nul : chacun de leurs mots-clés n'apparaît que dans ce film, et `min_df=2` les a tous retirés. Ces films ne sont comparés aux autres que par leurs genres et leurs valeurs numériques.

> **Objectif** : créer un nouveau modèle KNN sur `X2`, cette fois avec la distance cosinus (`metric="cosine"`) : elle compare la direction des vecteurs, pas leur longueur. Un film décrit par beaucoup de mots-clés n'est donc pas éloigné des autres pour cette seule raison.

In [133]:
modele2 = NearestNeighbors(n_neighbors=6, metric="cosine")
modele2.fit(X2)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


> **Objectif** : définir `recommander2`, qui interroge la version 2 grâce à la fonction générique `recommander` de la partie 4, puis une fonction `comparer` qui affiche côte à côte les titres recommandés par les deux versions. On ne compare pas les distances : la version 1 utilise la distance euclidienne, la version 2 la distance cosinus, leurs valeurs ne sont pas sur la même échelle.

In [134]:
def recommander2(tconst):
    return recommander(tconst, X2, modele2)

In [135]:
def comparer(tconst):
    v1 = recommander1(tconst)
    v2 = recommander2(tconst)
    return pd.DataFrame({"version 1": v1["title"].values,
                         "version 2": v2["title"].values})

> **Objectif** : comparer les versions 1 et 2 sur Alien. Dans la version 1, les suites d'Alien n'apparaissaient pas parmi ses voisins.

In [136]:
chercher("Alien")

,tconst,title,startYear,genres
294,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
646,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1053,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1078,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1508,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2215,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
3017,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4265,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"
4582,tt2316204,Alien: Covenant,2017,"Horror,Sci-Fi,Thriller"


In [137]:
comparer("tt0078748")    # Alien

,version 1,version 2
0,L'Invasion des profanateurs,Alien 3
1,28 semaines plus tard,Alien: Romulus
2,Chromosome 3,La Chose
3,Phantasm,Alien: Covenant
4,La Quatrième Dimension,L'Invasion des profanateurs


> **Constat** : la version 2 retrouve trois films de la saga (Alien 3, Alien: Romulus, Alien: Covenant), absents de la version 1, et La Chose, un autre huis clos avec une créature venue d'ailleurs. La version 1 ne proposait que des films aux mêmes genres (Horror, Sci-Fi) et de la même époque. Alien: Romulus arrive deuxième alors qu'il n'a que deux mots-clés (« alien » et « space ») : Alien possède ces deux mots, et la distance cosinus ne pénalise pas un film peu décrit, puisqu'elle ignore la longueur des vecteurs. En revanche, Aliens n'apparaît toujours pas : il diffère d'Alien sur trois genres (Action et Adventure d'un côté, Sci-Fi de l'autre).

> **Objectif** : creuser la saga Alien. Deux films de la saga manquent dans le top 5 de la version 2 (Aliens et Alien, la résurrection) : on cherche leur rang parmi les 30 plus proches voisins d'Alien, avec leurs genres et leur distance.

In [138]:
i_alien = films[films["tconst"] == "tt0078748"].index[0]
distances, indices = modele2.kneighbors(X2.iloc[[i_alien]], n_neighbors=30)
for rang in range(30):
    j = indices[0][rang]
    if "Alien" in films["title"].iloc[j]:
        print(rang, films["title"].iloc[j], films["genres"].iloc[j], round(distances[0][rang], 3))

0 Alien, le 8ème passager Horror,Sci-Fi 0.0
1 Alien 3 Action,Horror,Sci-Fi 0.382
2 Alien: Romulus Horror,Sci-Fi,Thriller 0.394
4 Alien: Covenant Horror,Sci-Fi,Thriller 0.407
6 Alien, la résurrection Action,Horror,Sci-Fi 0.429
20 Aliens, le retour Action,Adventure,Horror 0.472


> **Constat** : Alien, la résurrection est 6ᵉ, juste hors du top 5 : ses genres conviennent, mais il partage moins de mots-clés avec Alien. Aliens n'est que 20ᵉ, alors qu'il partage 7 mots-clés avec Alien, dont « xenomorph » (voir le début de la partie 5). La raison est dans ses genres : IMDb le classe en Action, Adventure, Horror, sans Sci-Fi. Il ne partage donc qu'un genre avec Alien (Horror, Sci-Fi) et en diffère sur trois, ce qui suffit, avec un poids de 0,7, à le repousser loin. Les genres pèsent encore beaucoup ; un poids plus faible pour les genres sera à essayer lors du réglage des poids.

> **Objectif** : comparer les versions 1 et 2 sur Taxi. Dans la version 1, Taxi 2 arrivait en tête.

In [139]:
chercher("Taxi")

,tconst,title,startYear,genres
1787,tt0152930,Taxi,1998,"Action,Comedy,Crime"
1873,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2547,tt0410730,Taxidermie,2006,"Comedy,Horror"
2795,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
5363,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
5756,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [140]:
comparer("tt0152930")    # Taxi

,version 1,version 2
0,Taxi 2,30 minutes maximum
1,Tueurs à gages,Mise à l'épreuve
2,Le Dernier Samaritain,2 Guns
3,La petite moutarde qui monte au nez,Super
4,Flic de haut vol,Flic de haut vol


> **Objectif** : comprendre pourquoi Taxi 2 disparaît : afficher les mots-clés de Taxi et de Taxi 2, et ceux qu'ils partagent.

In [141]:
mots_taxi = films[films["tconst"] == "tt0152930"]["keywords"].iloc[0].split(", ")
mots_taxi2 = films[films["tconst"] == "tt0183869"]["keywords"].iloc[0].split(", ")
print("Taxi :", mots_taxi)
print("Taxi 2 :", mots_taxi2)

communs_taxi = []
for mot in mots_taxi:
    if mot in mots_taxi2:
        communs_taxi.append(mot)
print("En commun :", communs_taxi)

Taxi : ['taxi', 'taxi driver', 'marseille', 'france', 'pizza boy', 'pizza delivery', 'police officer', 'bank robbery']
Taxi 2 : ['paris', 'france', 'yakuza', 'police operation', 'special unit']
En commun : ['france']


> **Constat** : en version 2, Taxi 2 disparaît des recommandations de Taxi, au profit de comédies policières américaines (30 minutes maximum, 2 Guns…). Les deux films ne partagent qu'un mot-clé, « france » : les mots-clés de Taxi 2 décrivent son intrigue, pas ce qui le relie au premier film. Les films américains partagent probablement avec Taxi des mots-clés plus précis, liés à son intrigue (livraison de pizzas, braquage, police). Rien, dans la matrice X2, n'indique que Taxi est un film français : c'est un défaut de la version 2.

### Modèle version 3

> **Objectif** : ajouter un septième bloc, la langue originale du film (colonne `original_language`, déjà présente dans le catalogue), pour que le moteur sache par exemple que Taxi est un film français. Chaque langue devient une colonne 0/1 : un film n'a qu'une langue, c'est donc du *one-hot* classique (contrairement aux genres, en *multi-hot*).

In [142]:
langues = pd.get_dummies(films["original_language"].fillna("inconnue")).astype(int)
langues.shape

(6089, 50)

> **Constat** : le catalogue compte 50 langues originales différentes : le bloc langue ajoute donc 50 colonnes, une par langue. Chaque ligne contient un seul 1, dans la colonne de la langue du film.

> **Objectif** : voir comment les films se répartissent entre les langues.

In [143]:
films["original_language"].value_counts().head(10)

original_language
en    4654
fr     214
hi     150
ja     149
es      97
ko      67
it      66
cn      62
ta      61
de      57
Name: count, dtype: int64

> **Constat** : les langues sont très inégalement représentées : 4 654 films sur 6 089, plus des trois quarts du catalogue, sont en anglais (en). Viennent ensuite, loin derrière, le français (214 films), l'hindi (150) et le japonais (149). La colonne « en » ne distinguera donc pas les films anglophones entre eux, mais le bloc langue devrait aider les films des autres langues, comme Taxi, à retrouver des films de leur langue.

> **Objectif** : assembler la matrice `X3` : les six blocs de la version 2, plus la langue avec un poids de 0,8. On crée ensuite le modèle et la fonction `recommander3`, comme pour la version 2.

In [144]:
X3 = np.hstack([0.7 * genres.values,
                num_ponderes.values,
                mots.toarray(),
                0.8 * langues.values])
X3 = pd.DataFrame(X3)
X3.shape

(6089, 6551)

In [145]:
modele3 = NearestNeighbors(n_neighbors=6, metric="cosine")
modele3.fit(X3)

def recommander3(tconst):
    return recommander(tconst, X3, modele3)

> **Objectif** : réécrire `comparer` pour afficher côte à côte les recommandations des trois versions, puis retester Taxi.

In [146]:
def comparer(tconst):
    v1 = recommander1(tconst)
    v2 = recommander2(tconst)
    v3 = recommander3(tconst)
    return pd.DataFrame({"version 1": v1["title"].values,
                         "version 2": v2["title"].values,
                         "version 3": v3["title"].values})

In [147]:
chercher("Taxi")

,tconst,title,startYear,genres
1787,tt0152930,Taxi,1998,"Action,Comedy,Crime"
1873,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2547,tt0410730,Taxidermie,2006,"Comedy,Horror"
2795,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
5363,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
5756,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [148]:
comparer("tt0152930")

,version 1,version 2,version 3
0,Taxi 2,30 minutes maximum,Taxi 2
1,Tueurs à gages,Mise à l'épreuve,Dobermann
2,Le Dernier Samaritain,2 Guns,Micmacs à tire-larigot
3,La petite moutarde qui monte au nez,Super,La petite moutarde qui monte au nez
4,Flic de haut vol,Flic de haut vol,"Bon Cop, Bad Cop"


> **Constat** : avec la langue, Taxi 2 revient en tête, suivi de films en français : Dobermann, Micmacs à tire-larigot, La petite moutarde qui monte au nez, et Bon Cop, Bad Cop (film québécois en partie francophone). Le reste de la saga n'apparaît pas, mais `chercher("Taxi")` montre qu'il n'est pas dans le catalogue : Taxi 3 (note IMDb 5,8), Taxi 4 (5,6) et Taxi 5 (4,6) sont tous sous le seuil de 6 du filtre de l'équipe (source : imdb.com). La version 3 retrouve donc toute la saga disponible.

In [149]:
chercher("Alien")

,tconst,title,startYear,genres
294,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
646,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1053,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1078,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1508,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2215,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
3017,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4265,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"
4582,tt2316204,Alien: Covenant,2017,"Horror,Sci-Fi,Thriller"


In [150]:
comparer("tt0078748")

,version 1,version 2,version 3
0,L'Invasion des profanateurs,Alien 3,Alien 3
1,28 semaines plus tard,Alien: Romulus,La Chose
2,Chromosome 3,La Chose,Alien: Romulus
3,Phantasm,Alien: Covenant,L'Invasion des profanateurs
4,La Quatrième Dimension,L'Invasion des profanateurs,Alien: Covenant


In [151]:
recommander2("tt0078748")

,tconst,title,startYear,genres,distance
1078,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi",0.382021
4265,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller",0.393524
465,tt0084787,La Chose,1982,"Horror,Mystery,Sci-Fi",0.394142
4582,tt2316204,Alien: Covenant,2017,"Horror,Sci-Fi,Thriller",0.406925
279,tt0077745,L'Invasion des profanateurs,1978,"Horror,Sci-Fi",0.413840


> **Constat** : pour Alien, la version 3 propose les mêmes cinq films que la version 2 ; seul l'ordre bouge (La Chose passe deuxième), à cause de quasi-égalités : en version 2, Alien: Romulus et La Chose étaient déjà presque à la même distance d'Alien. Ces films sont tous en anglais, comme les trois quarts du catalogue : la colonne « en » vaut 1 pour tous et ne les distingue pas entre eux. La langue aide les films non anglophones, comme Taxi, sans dégrader les autres.

> **Objectif** : comparer les trois versions sur six autres films tests, connus de tous et de genres variés : Toy Story, La Communauté de l'anneau, Matrix (anglophones), puis Astérix & Obélix : Mission Cléopâtre, Intouchables et Amélie (francophones).

In [152]:
chercher("toy")

,tconst,title,startYear,genres
1406,tt0114709,Toy Story,1995,"Adventure,Animation,Comedy"
1609,tt0120363,Toy Story 2,1999,"Adventure,Animation,Comedy"
2645,tt0435761,Toy Story 3,2010,"Adventure,Animation,Comedy"
4348,tt1979376,Toy Story 4,2019,"Adventure,Animation,Comedy"
4913,tt29355505,Toy Story 5,2026,"Adventure,Animation,Comedy"
5388,tt4562518,Citoyen d'honneur,2016,"Comedy,Drama"


> **Constat** : la version 1 propose des films d'animation de la même époque (Monstres & Cie, Shrek, Aladdin…), avec Toy Story 2 en 3ᵉ position seulement. Les versions 2 et 3 placent en tête les trois suites (Toy Story 4, 3 et 2), puis deux autres films Pixar (Monstres & Cie, Vice-versa). Nette amélioration ; la langue ne change rien, tous ces films étant anglophones.

In [153]:
comparer("tt0114709")

,version 1,version 2,version 3
0,Monstres & Cie,Toy Story 4,Toy Story 4
1,Shrek,Toy Story 3,Toy Story 3
2,Toy Story 2,Toy Story 2,Toy Story 2
3,Aladdin,Monstres & Cie,Monstres & Cie
4,Le Monde de Nemo,Vice-versa,Vice-versa


In [154]:
chercher("anneau")

,tconst,title,startYear,genres
280,tt0077869,Le Seigneur des anneaux,1978,"Adventure,Animation,Fantasy"
1638,tt0120737,Le Seigneur des anneaux : La Communauté de l'a...,2001,"Adventure,Drama,Fantasy"
1835,tt0167260,Le Seigneur des anneaux : Le Retour du roi,2003,"Adventure,Drama,Fantasy"
1836,tt0167261,Le Seigneur des anneaux : Les Deux Tours,2002,"Adventure,Drama,Fantasy"
3869,tt14824600,Le Seigneur des Anneaux : La Guerre des Rohirrim,2024,"Action,Adventure,Animation"
5474,tt5027774,3 Billboards : Les Panneaux de la vengeance,2017,"Comedy,Crime,Drama"
6038,tt9376612,Shang-Chi et la Légende des Dix Anneaux,2021,"Action,Adventure,Fantasy"


In [155]:
comparer("tt0120737")

,version 1,version 2,version 3
0,Le Seigneur des anneaux : Les Deux Tours,Le Seigneur des anneaux : Le Retour du roi,Le Seigneur des anneaux : Le Retour du roi
1,Le Seigneur des anneaux : Le Retour du roi,Le Seigneur des anneaux : Les Deux Tours,Le Seigneur des anneaux : Les Deux Tours
2,Big Fish,Le Hobbit : La Désolation de Smaug,Le Hobbit : La Désolation de Smaug
3,The Fall: Le voyage imaginaire,Le Hobbit: La Bataille des Cinq Armées,Le Hobbit: La Bataille des Cinq Armées
4,Excalibur,Excalibur,Excalibur


> **Constat** : les trois versions retrouvent les deux suites. Les versions 2 et 3 remplacent Big Fish et The Fall, deux films de fantasy sans lien avec la saga, par les deux films Le Hobbit, qui se déroulent dans le même univers. Amélioration ; versions 2 et 3 identiques.

In [156]:
chercher("matrix")

,tconst,title,startYear,genres
1728,tt0133093,Matrix,1999,"Action,Sci-Fi"
1970,tt0234215,Matrix Reloaded,2003,"Action,Sci-Fi"
1987,tt0242653,Matrix Revolutions,2003,"Action,Sci-Fi"


In [157]:
comparer("tt0133093")

,version 1,version 2,version 3
0,Avengers,Matrix Reloaded,Matrix Reloaded
1,X-Men : Le Commencement,Matrix Revolutions,Matrix Revolutions
2,Captain America: Civil War,Avengers,Avengers
3,Matrix Reloaded,Captain America: Civil War,Captain America: Civil War
4,The Amazing Spider-Man,X-Men : Le Commencement,X-Men : Le Commencement


> **Constat** : la version 1 propose surtout des films de super-héros (Avengers, X-Men, Captain America…), avec Matrix Reloaded en 4ᵉ position seulement. Les versions 2 et 3 placent en tête les deux suites, Matrix Reloaded et Matrix Revolutions. Matrix Resurrections, absent du catalogue, ne peut pas apparaître. Amélioration ; versions 2 et 3 identiques.

In [158]:
chercher("astérix")

,tconst,title,startYear,genres
166,tt0072901,Les 12 Travaux d'Astérix,1976,"Adventure,Animation,Comedy"
577,tt0088748,Astérix et la Surprise de César,1985,"Adventure,Animation,Comedy"
649,tt0090667,Astérix chez les Bretons,1986,"Adventure,Animation,Comedy"
1731,tt0133385,Astérix & Obélix contre César,1999,"Adventure,Comedy,Family"
2010,tt0250223,Astérix & Obélix : Au service de Cléopâtre,2002,"Adventure,Comedy,Family"
2398,tt0371552,Astérix et les Vikings,2006,"Adventure,Animation,Comedy"
5245,tt3759416,Astérix : Le Domaine des dieux,2014,"Adventure,Animation,Comedy"
5900,tt8001346,Astérix: Le Secret de la potion magique,2018,"Adventure,Animation,Comedy"


In [159]:
comparer("tt0250223")

,version 1,version 2,version 3
0,Le petit monde de Charlotte,Aladdin,Astérix & Obélix contre César
1,Bean,Paddington 2,Le Petit Nicolas
2,Major Payne,Astérix & Obélix contre César,Ernest et Célestine
3,"Les Muppets, le retour",Paddington,Astérix et la Surprise de César
4,Les Désastreuses Aventures des orphelins Baude...,Le Retour de Mary Poppins,Les Aventures de Rabbi Jacob


> **Constat** : la version 1 ne propose aucun Astérix, seulement des comédies familiales anglophones de la même époque. La version 2 retrouve Astérix & Obélix contre César, l'autre film en prises de vue réelles, en 3ᵉ position seulement, parmi des films familiaux anglophones (Aladdin, Paddington…). La version 3 le place en tête et ajoute un dessin animé de la saga, Astérix et la Surprise de César, puis des classiques familiaux français (Le Petit Nicolas, Ernest et Célestine, Les Aventures de Rabbi Jacob). Amélioration nette, mais la saga reste incomplète : 2 des 7 autres films Astérix seulement. Les dessins animés ont sans doute des genres (Animation) et des mots-clés trop différents de ceux d'un film en prises de vue réelles. Un fichier de sagas fait main réglerait ce cas.

In [160]:
chercher("Intouchables")

,tconst,title,startYear,genres
4122,tt1675434,Intouchables,2011,"Comedy,Drama"


In [161]:
comparer("tt1675434")

,version 1,version 2,version 3
0,The Grand Budapest Hotel,Dasvi,Le grand bain
1,The Truman Show,Dog,C.R.A.Z.Y.
2,Birdman ou (La Surprenante vertu de l'ignorance),El buen patrón,Demain tout commence
3,Little Miss Sunshine,The Upside - Seconde Chance,Monsieur Lazhar
4,Lost in Translation,Once Upon a Time in... Hollywood,Monsieur Ibrahim et les fleurs du Coran


> **Constat** : la version 1 propose des comédies dramatiques anglophones (The Grand Budapest Hotel, The Truman Show…). La version 2 mélange des films de pays variés (Dasvi, El buen patrón…) et retrouve The Upside, le remake américain d'Intouchables. La version 3 ne propose plus que des films francophones (Le grand bain, Demain tout commence, et deux films québécois, C.R.A.Z.Y. et Monsieur Lazhar), mais elle perd The Upside. Résultat mitigé : la langue rapproche des films par leur langue, pas forcément par leur histoire.

In [162]:
chercher("Amélie")

,tconst,title,startYear,genres
1932,tt0211915,Amélie des Abbesses,2001,"Comedy,Romance"


In [163]:
comparer("tt0211915")

,version 1,version 2,version 3
0,"40 ans, toujours puceau",Un accord parfait,L'arnacoeur
1,Mary à tout prix,"Hitch, expert en séduction",Hors de prix
2,Serial noceurs,Premalu,Bienvenue chez les Ch'tis
3,Pretty Woman,Mary à tout prix,Lol
4,Mon beau-père et moi,Mon beau-père et moi,Les émotifs anonymes


> **Constat** : les versions 1 et 2 proposent surtout des comédies romantiques américaines (Mary à tout prix, Hitch…). La version 3 ne propose que des comédies françaises (L'arnacoeur, Hors de prix, Bienvenue chez les Ch'tis, LOL, Les émotifs anonymes), plus proches du public d'Amélie, mais rarement de son univers poétique.
>
> **Remarque** : dans le catalogue, le titre français d'Amélie est « Amélie des Abbesses » au lieu de « Le Fabuleux Destin d'Amélie Poulain » : le choix du titre français est à revoir avec l'équipe.

### Bilan des trois versions

> **Constat** : sur les huit films tests, la comparaison des trois versions donne :
>
> | Film | Version 1 | Version 2 | Version 3 |
> |---|---|---|---|
> | Alien | aucune suite | 3 films de la saga | 3 films de la saga |
> | Taxi | Taxi 2 en tête | Taxi 2 disparaît | Taxi 2 en tête, films français |
> | Toy Story | Toy Story 2 seulement | les 3 suites | les 3 suites |
> | La Communauté de l'anneau | 2 suites, fantasy | 2 suites, 2 Hobbit | 2 suites, 2 Hobbit |
> | Matrix | super-héros | les 2 suites en tête | les 2 suites en tête |
> | Intouchables | comédies anglophones | dont son remake | films francophones |
> | Amélie | comédies américaines | comédies américaines | comédies françaises |
> | Mission Cléopâtre | aucun Astérix | 1 Astérix | 2 Astérix en tête |
>
> Les mots-clés (version 2) font retrouver les sagas et les univers, ce que les genres seuls ne permettaient pas. La langue (version 3) corrige les films non anglophones, comme Taxi ou Astérix, sans changer les films anglophones ; mais elle peut aussi rapprocher des films seulement parce qu'ils partagent une langue (Intouchables perd son remake).
>
> **Décision** : retenir la version 3 comme base du moteur.

### Réglage des poids

> **Objectif** : tester l'importance de la durée. On reprend la version 3 en mettant le poids de la durée à 0, sans changer les autres poids, puis on compare les recommandations avec celles de la version 3.

In [164]:
poids_num

{'startYear': 0.7,
 'runtimeMinutes': 0.7,
 'averageRating': 0.7,
 'log_votes': 0.7}

In [165]:
poids_sans_duree = poids_num.copy()
poids_sans_duree["runtimeMinutes"] = 0

num_sans_duree = numeriques_norm.copy()
for col in poids_sans_duree:
    num_sans_duree[col] = numeriques_norm[col] * poids_sans_duree[col]

X3_sans_duree = pd.DataFrame(np.hstack([0.7 * genres.values,
                                        num_sans_duree.values,
                                        mots.toarray(),
                                        0.8 * langues.values]))
modele3_sans_duree = NearestNeighbors(n_neighbors=6, metric="cosine")
modele3_sans_duree.fit(X3_sans_duree)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


> **Objectif** : comparer, sur les huit films tests, les recommandations de la version 3 et de la variante sans durée.

In [166]:
def recommander3_sans_duree(tconst):
    return recommander(tconst, X3_sans_duree, modele3_sans_duree)

def comparer_duree(tconst):
    v3 = recommander3(tconst)
    v3_sans = recommander3_sans_duree(tconst)
    return pd.DataFrame({"version 3": v3["title"].values,
                         "sans durée": v3_sans["title"].values})

> **Remarque** : dans une cellule, seul le résultat de la dernière ligne s'affiche automatiquement en tableau. Dans une boucle, il faut donc appeler soi-même `display`, une fonction de Jupyter disponible sans import, pour afficher chaque DataFrame sous sa forme habituelle ; `print` l'afficherait en texte brut.

In [167]:
films_tests = {"Alien": "tt0078748", "Taxi": "tt0152930",
               "Toy Story": "tt0114709", "La Communauté de l'anneau": "tt0120737",
               "Matrix": "tt0133093", "Intouchables": "tt1675434",
               "Amélie": "tt0211915", "Mission Cléopâtre": "tt0250223"}

for titre in films_tests:
    print(titre)
    display(comparer_duree(films_tests[titre]))

Alien


,version 3,sans durée
0,Alien 3,Alien 3
1,La Chose,La Chose
2,Alien: Romulus,Alien: Romulus
3,L'Invasion des profanateurs,L'Invasion des profanateurs
4,Alien: Covenant,Alien: Covenant


Taxi


,version 3,sans durée
0,Taxi 2,Taxi 2
1,Dobermann,Dobermann
2,Micmacs à tire-larigot,Micmacs à tire-larigot
3,La petite moutarde qui monte au nez,La petite moutarde qui monte au nez
4,"Bon Cop, Bad Cop","Bon Cop, Bad Cop"


Toy Story


,version 3,sans durée
0,Toy Story 4,Toy Story 4
1,Toy Story 3,Toy Story 3
2,Toy Story 2,Toy Story 2
3,Monstres & Cie,Monstres & Cie
4,Vice-versa,Vice-versa


... (6 outputs left out)

Amélie


,version 3,sans durée
0,L'arnacoeur,L'arnacoeur
1,Hors de prix,Les émotifs anonymes
2,Bienvenue chez les Ch'tis,Hors de prix
3,Lol,Bienvenue chez les Ch'tis
4,Les émotifs anonymes,20 ans d'écart


Mission Cléopâtre


,version 3,sans durée
0,Astérix & Obélix contre César,Astérix & Obélix contre César
1,Le Petit Nicolas,Le Petit Nicolas
2,Ernest et Célestine,Ernest et Célestine
3,Astérix et la Surprise de César,Astérix et la Surprise de César
4,Les Aventures de Rabbi Jacob,Les 12 Travaux d'Astérix


> **Constat** : sans la durée, 4 films sur 8 ont exactement les mêmes recommandations (Alien, Taxi, Toy Story, Intouchables). Pour les 4 autres, seule la fin de liste change : le 5ᵉ voisin de La Communauté de l'anneau (Excalibur remplacé par Big Fish), de Matrix (X-Men : Le Commencement remplacé par Terminator 3) et de Mission Cléopâtre (Les Aventures de Rabbi Jacob remplacé par Les 12 Travaux d'Astérix, un troisième film de la saga), et l'ordre des comédies proposées pour Amélie. La durée pèse donc très peu face aux mots-clés, aux genres et à la langue.

> **Décision** : mettre le poids de la durée à 0 dans le moteur final. Les autres poids restent ceux de la version 3 (genres 0,7, année, note et popularité 0,7, mots-clés 1, langue 0,8).

## 6. Du notebook à l'appli

> **Objectif** : réunir le moteur retenu (version 3, durée à 0) dans un fichier `reco.py`, sous forme de fonctions que l'appli Streamlit pourra importer. Le notebook a servi à explorer et à décider ; `reco.py` ne garde que le nécessaire.

> **Objectif** : vérifier que `reco.py` reproduit exactement le moteur retenu. On recharge le catalogue depuis le fichier CSV, comme le fera l'appli, puis on prépare les caractéristiques et le modèle avec les fonctions de `reco.py`.

In [168]:
import sys
sys.path.append("..")
import reco

films_reco = pd.read_csv("../data/films_prepares.csv")
X_reco = reco.preparer_caracteristiques(films_reco)
modele_reco = reco.creer_modele(X_reco)
X_reco.shape

(6089, 6551)

> **Objectif** : comparer, pour les huit films tests, les recommandations de `reco.py` et celles de la variante sans durée du notebook.

In [169]:
for titre in films_tests:
    tconst = films_tests[titre]
    titres_reco = list(reco.recommander(tconst, films_reco, X_reco, modele_reco)["title"])
    titres_notebook = list(recommander3_sans_duree(tconst)["title"])
    print(titre, ":", titres_reco == titres_notebook)

Alien : True
Taxi : True
Toy Story : True
La Communauté de l'anneau : True
Matrix : True
Intouchables : True
Amélie : True
Mission Cléopâtre : True


> **Constat** : pour les huit films tests, `reco.py` donne exactement les mêmes recommandations que la variante sans durée du notebook. Le moteur est désormais réuni dans trois fonctions (`preparer_caracteristiques`, `creer_modele`, `recommander`), prêtes à être importées par l'appli.

> **Remarque** : le notebook garde en mémoire la version de `reco.py` chargée par `import reco`. Après une modification de `reco.py`, il faut redémarrer le noyau, puis réexécuter les cellules.